# Feature Visualization Video

指定した元動画に生flow動画と、flow/信頼度重み付きflowの3x3グラフ・広域振動スコアを出力します。

`OUTPUT_SETTINGS` の18項目だけで出力を切り替えます。3x3グラフには振動スコアの背景オーバーレイを常に描画します。


## 処理の流れ

1. 対象sampleのfront/rear動画を選び、共通flow cacheを読み込みます。
2. cacheがない場合はフレーム単位flowを測定し、孤立した静止フレームを補正してから0.1秒binへ按分します。
3. raw flow動画、3x3グラフ、広域振動スコア画像を必要なものだけ出力します。


In [15]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - Path / OpenCV / pandas / tqdm など、可視化動画生成に必要なライブラリを読み込みます。
# - tqdm がない環境でも処理本体を動かせるよう、最低限のダミークラスを用意します。
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
import importlib
import shutil
import subprocess
import sys
from typing import Any

import cv2
import numpy as np
import pandas as pd


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "forklift_features").exists():
            return candidate
    raise FileNotFoundError(f"Could not find src/forklift_features from {start}.")


_IMPORT_ROOT = _find_import_root()
src_path = str(_IMPORT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import cache as feature_cache
from forklift_features import content_update
from forklift_features import flow_extract
from forklift_features import flow_tensor
from forklift_features import vibration as feature_vibration
feature_cache = importlib.reload(feature_cache)
content_update = importlib.reload(content_update)
flow_extract = importlib.reload(flow_extract)
flow_tensor = importlib.reload(flow_tensor)
feature_vibration = importlib.reload(feature_vibration)
from forklift_features.paths import find_project_root, safe_path_part

try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        def __init__(self, total=None, desc=None):
            self.total = total
            self.desc = desc
        def __enter__(self):
            progress = self
            return progress
        def __exit__(self, exc_type, exc, tb):
            suppress_exception = False
            return suppress_exception
        def update(self, n=1):
            pass


In [ ]:
# ------------------------------------------------------------
# セル概要: 対象動画と可視化設定
# ------------------------------------------------------------
# - PROJECT_ROOT、入出力先、可視化したい sample_id/view、描画レイヤー、Farneback 設定をまとめます。
# - 前処理済み動画を探し、出力動画とCSVの保存先を決める関数もここで定義します。
# ------------------------------------------------------------

PROJECT_ROOT = find_project_root()
MOVIE_PREPROCESS_DIR = PROJECT_ROOT / "data" / "movie_preprocess"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "feature_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ここで可視化したい動画を sample_id で複数指定します。
# 指定した sample_id は front/rear の両方を出力します。
# 空配列にすると data/movie_preprocess 配下の全 front/rear ペアを処理します。
VIDEO_TARGETS = [
    # "1001",
    "032_停車中に車両に衝突される",
    "060_後進時、ラックに衝突",
]


# 関数メモ: find_movie_preprocess_video
# - sample_id と front/rear view から、前処理済み mp4 の実パスを探します。
# - 複数ヒットした場合は曖昧さを例外にし、違う動画を可視化してしまう事故を防ぎます。

def find_movie_preprocess_video(
    sample_id: str,
    view: str,
    movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR,
) -> Path:
    view = view.lower().strip()
    if view not in {"front", "rear"}:
        raise ValueError(f"view must be 'front' or 'rear': {view}")
    filename = f"{sample_id}_{view}.mp4"
    candidates = sorted(movie_preprocess_dir.glob(f"*/*/{filename}"))
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {filename} under {movie_preprocess_dir}"
        )
    if len(candidates) > 1:
        raise ValueError("Multiple matching videos were found. Please make sample_id unique or narrow the search logic:\n" + "\n".join(str(p) for p in candidates))
    return candidates[0]


# 関数メモ: list_movie_preprocess_video_pairs
# - movie_preprocess 配下の front/rear mp4 ペアを列挙します。
# - VIDEO_TARGETS が空の場合は、この一覧をそのまま処理対象にします。

def list_movie_preprocess_video_pairs(movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> list[dict[str, Any]]:
    pairs = []
    for front_path in sorted(movie_preprocess_dir.glob("*/*/*_front.mp4")):
        if not front_path.is_file():
            continue
        sample_id = front_path.name[:-len("_front.mp4")]
        rear_path = front_path.with_name(f"{sample_id}_rear.mp4")
        if not rear_path.is_file():
            print(f"skipped target without rear video: {front_path}")
            continue
        pairs.append({
            "sample_id": sample_id,
            "front_video_path": front_path,
            "rear_video_path": rear_path,
        })
    return pairs


# 関数メモ: build_video_targets
# - VIDEO_TARGETS が空なら全動画、指定があれば各 sample_id の front/rear ペアを作ります。

def build_video_targets(video_targets: list[Any]) -> list[dict[str, Any]]:
    if not video_targets:
        return list_movie_preprocess_video_pairs()
    targets = []
    seen_sample_ids = set()
    for target in video_targets:
        sample_id = str(target["sample_id"] if isinstance(target, dict) else target)
        if sample_id in seen_sample_ids:
            continue
        seen_sample_ids.add(sample_id)
        front_video_path = find_movie_preprocess_video(sample_id, "front")
        rear_video_path = find_movie_preprocess_video(sample_id, "rear")
        targets.append({
            "sample_id": sample_id,
            "front_video_path": front_video_path,
            "rear_video_path": rear_video_path,
        })
    return targets


OUTPUT_FILE_ORDER = [
    "raw_flow_video",
    "flow_x",
    "flow_y",
    "vector_change",
    "flow_x_broad_vib_score",
    "flow_y_broad_vib_score",
    "vector_change_broad_vib_score",
    "t_flow_x",
    "t_flow_y",
    "t_vector_change",
    "t_flow_x_broad_vib_score",
    "t_flow_y_broad_vib_score",
    "t_vector_change_broad_vib_score",
    "motion_impulse_score",
    "accel_impact_x_score",
    "accel_impact_y_score",
    "t_accel_impact_x_score",
    "t_accel_impact_y_score",
    "flow_edge_confidence",
    "flow_coherence_confidence",
    "flow_backward_consistency",
    "flow_measurement_confidence",
]
OUTPUT_FILE_NUMBERS = {key: f"{index:02d}" for index, key in enumerate(OUTPUT_FILE_ORDER)}


def numbered_output_stem(sample_id: str, view: str, output_key: str, suffix: str | None = None) -> str:
    number = OUTPUT_FILE_NUMBERS[output_key]
    tail = safe_path_part(suffix or output_key)
    return f"{safe_path_part(sample_id)}_{view}_{number}_{tail}"


def numbered_output_path(sample_dir: Path, sample_id: str, view: str, output_key: str, extension: str, suffix: str | None = None) -> Path:
    extension = extension.lstrip(".")
    return sample_dir / f"{numbered_output_stem(sample_id, view, output_key, suffix=suffix)}.{extension}"


# 関数メモ: build_output_paths
# - 入力動画と sample_id/view から、動画ごとの出力ディレクトリ、可視化動画、特徴CSVの保存先を作ります。
# - 同じ sample_id の front/rear 動画とグラフを1つのディレクトリへまとめます。
def build_output_paths(input_video_path: Path, sample_id: str, view: str) -> tuple[Path, Path, Path]:
    try:
        category, environment = input_video_path.relative_to(MOVIE_PREPROCESS_DIR).parts[:2]
    except ValueError:
        category, environment = "unknown", "unknown"
    output_dir = OUTPUT_DIR / f"{safe_path_part(sample_id)}_{safe_path_part(category)}_{safe_path_part(environment)}"
    output_stem = numbered_output_stem(sample_id, view, "raw_flow_video", suffix=f"{safe_path_part(category)}_{safe_path_part(environment)}_grid_flow")
    return output_dir, output_dir / f"{output_stem}.mp4", output_dir / f"{output_stem}_features.csv"


VIDEO_TARGET_PAIRS = build_video_targets(VIDEO_TARGETS)

# 出力の ON/OFF。ここにある項目だけをファイル出力対象にします。
OUTPUT_SETTINGS = {
    "raw_flow_video": True,
    "vector_change": False,
    "flow_x": True,
    "flow_y": True,
    "vector_change_broad_vib_score": False,
    "flow_x_broad_vib_score": False,
    "flow_y_broad_vib_score": False,
    "t_vector_change": False,
    "t_flow_x": True,
    "t_flow_y": True,
    "t_vector_change_broad_vib_score": False,
    "t_flow_x_broad_vib_score": True,
    "t_flow_y_broad_vib_score": True,
    "motion_impulse_score": False,
    "accel_impact_x_score": False,
    "accel_impact_y_score": False,
    "t_accel_impact_x_score": True,
    "t_accel_impact_y_score": True,
    "flow_edge_confidence": False,
    "flow_coherence_confidence": False,
    "flow_backward_consistency": True,
    "flow_measurement_confidence": False,
}
OVERWRITE_GRAPH_OUTPUTS = True
VIBRATION_SCORE_CONFIG = {
    "window_sec": 0.5,
    "hop_sec": 0.2,
    "alpha_min": 0.04,
    "alpha_max": 0.42,
    "high_ratio_fraction": 0.25,
    "broad_vib_score_weights": {"A": 0.25, "B": 0.75, "C": 0.0},
    "broad_vib_low_intensity_percentile": 20.0,
    "broad_vib_spread_power": 1.0,
}
MOTION_IMPULSE_SCORE_CONFIG = {
    "intensity_percentile": 75.0,
    "intensity_upper_percentile": 95.0,
    "min_vector_change": 1e-6,
    "spread_power": 0.7,
    "direction_power": 0.7,
    "intensity_gate_power": 0.5,
    "component_weights": {"intensity": 0.15, "spread": 0.45, "direction": 0.40},
}
ACCEL_IMPACT_SCORE_CONFIG = {
    "x_clip_abs": 1.0,
    "y_clip_abs": 1.0,
    "t_x_clip_abs": 1.0,
    "t_y_clip_abs": 1.0,
    "eps": 1e-6,
}

VISUALIZATION_CONFIG: dict[str, Any] = {
    "resize_width": 960,
    "flow_sample_sec": 0.1,
    "flow_reliable_error_threshold_px": 1.0,
    "t_flow_reliable_weight_low": 0.35,
    "t_flow_reliable_weight_high": 0.85,
    "t_flow_reliable_weight_gamma": 2.0,
    "flow_static_frame_max_grid_vector_mag": 0.02,
    "max_frames": None,
    "fourcc": "mp4v",
    "include_audio": True,
    "browser_compatible_mp4": True,
    "keep_intermediate_video": False,
    "ffmpeg_video_codec": "libx264",
    "ffmpeg_audio_codec": "aac",
    "ffmpeg_audio_bitrate": "128k",
    "ffmpeg_crf": 20,
    "ffmpeg_preset": "medium",
    "grid": {
        "shape": (3, 3),
        "line_color": (80, 220, 255),
        "line_thickness": 1,
    },
    "flow_arrows": {
        "color": (40, 255, 80),
        "tip_color": (0, 180, 255),
        "outline_color": (0, 0, 0),
        "outline_extra_thickness": 4,
        "thickness": 2,
        "tip_length": 0.25,
        "scale": 12.0,
        "max_length_ratio": 0.42,
        "min_magnitude": 0.02,
        "hold_below_min_magnitude": False,
        "show_values": True,
        "value_label_color": (240, 245, 255),
        "value_label_outline_color": (0, 0, 0),
        "value_label_font_scale": 0.42,
        "value_label_thickness": 1,
        "value_label_outline_extra_thickness": 2,
        "value_label_offset": (6, 18),
        "value_label_line_gap": 16,
        "value_label_decimals": 2,
    },
    "farneback": {
        "pyr_scale": 0.5,
        "levels": 3,
        "winsize": 15,
        "iterations": 3,
        "poly_n": 5,
        "poly_sigma": 1.2,
        "flags": 0,
    },
}

# 学習・推論 notebook と同じ動画単位 cache を、可視化でも読み取りに使います。
ANOMALY_FEATURE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_feature_poc"
FEATURE_CACHE_DIR = ANOMALY_FEATURE_OUTPUT_DIR / "sample_feature_cache"
ENABLE_SAMPLE_FEATURE_CACHE = True
FEATURE_CACHE_VERSION = "sample_feature_cache_v12"
FLOW_CACHE_VERSION = "sample_flow_cache_v14"
FLOW_CACHE_METHOD = "farneback_frame_flow_isolated_static_split_0p1s"
SHARED_CACHE_FRAME_RESIZE_WIDTH = 480
SHARED_CACHE_WINDOW_SEC = 0.2
SHARED_CACHE_AUDIO_SR = 16000
SHARED_CACHE_N_MELS = 16
SHARED_CACHE_FLOW_ANALYSIS_SCALE = 0.75
SHARED_CACHE_BROAD_VIBRATION_FEATURE_NAMES = [
    "vector_change",
    "flow_x",
    "flow_y",
    "t_vector_change",
    "t_flow_x",
    "t_flow_y",
]
FEATURE_OUTPUTS = {"grid_flow": True}

print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"MOVIE_PREPROCESS  : {MOVIE_PREPROCESS_DIR}")
print(f"video targets     : {len(VIDEO_TARGET_PAIRS)}")
for target in VIDEO_TARGET_PAIRS:
    print(f"- {target['sample_id']} / front,rear")
print(f"OUTPUT_SETTINGS   : {OUTPUT_SETTINGS}")


PROJECT_ROOT      : /workspaces/forklift
MOVIE_PREPROCESS  : /workspaces/forklift/data/movie_preprocess
video targets     : 2
- 032_停車中に車両に衝突される / front,rear
- 060_後進時、ラックに衝突 / front,rear
OUTPUT_SETTINGS   : {'raw_flow_video': True, 'vector_change': False, 'flow_x': False, 'flow_y': False, 'vector_change_broad_vib_score': False, 'flow_x_broad_vib_score': False, 'flow_y_broad_vib_score': False, 't_vector_change': False, 't_flow_x': True, 't_flow_y': True, 't_vector_change_broad_vib_score': False, 't_flow_x_broad_vib_score': True, 't_flow_y_broad_vib_score': True, 'motion_impulse_score': False, 'accel_impact_x_score': False, 'accel_impact_y_score': False, 't_accel_impact_x_score': True, 't_accel_impact_y_score': True, 'flow_edge_confidence': False, 'flow_coherence_confidence': False, 'flow_backward_consistency': True, 'flow_measurement_confidence': False}


In [17]:
# ------------------------------------------------------------
# セル概要: 動画読み書きとグリッド特徴
# ------------------------------------------------------------
# - 動画を開く、出力 writer を作る、ffmpeg でブラウザ互換MP4へ変換する関数群です。
# - optical flow の抽出・グリッド集約は src/forklift_features/flow_extract.py の共通実装を使います。
# ------------------------------------------------------------

def open_video(video_path: str | Path) -> cv2.VideoCapture:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    return capture


# 関数メモ: make_video_writer
# - 出力先ディレクトリを作成し、指定FPS・サイズの VideoWriter を返します。
# - 動画ファイルが作れない場合は RuntimeError にして、失敗を見逃さないようにします。

def make_video_writer(output_path: str | Path, fps: float, frame_shape: tuple[int, ...], fourcc_name: str = "mp4v") -> cv2.VideoWriter:
    h, w = frame_shape[:2]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*fourcc_name),
        float(fps),
        (int(w), int(h)),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not create video writer: {output_path}")
    return writer


# 関数メモ: build_intermediate_video_path
# - OpenCV 書き出し用の一時 mp4 パスを作ります。
# - 後で ffmpeg によるブラウザ互換変換を行う場合に、最終出力と中間出力を分けます。

def build_intermediate_video_path(output_path: str | Path) -> Path:
    output_path = Path(output_path)
    return output_path.with_name(f"{output_path.stem}_opencv_tmp{output_path.suffix}")


# 関数メモ: find_visualization_audio_source
# - front/rear 可視化動画に付ける音声を探します。
# - 前処理済みの `*_audio.wav` があれば優先し、なければ入力動画自身の音声を任意で使います。

def find_visualization_audio_source(input_video_path: str | Path) -> Path:
    input_video_path = Path(input_video_path)
    stem = input_video_path.stem
    if stem.endswith("_front") or stem.endswith("_rear"):
        sample_id = stem.rsplit("_", 1)[0]
        audio_path = input_video_path.with_name(f"{sample_id}_audio.wav")
        if audio_path.exists():
            return audio_path
    return input_video_path


# 関数メモ: transcode_to_browser_compatible_mp4
# - OpenCV で書いた mp4 を ffmpeg で H.264/yuv420p へ変換し、必要なら音声も合成します。
# - VSCode やブラウザでプレビューしやすい形式にするための後処理です。

def transcode_to_browser_compatible_mp4(
    input_path: str | Path,
    output_path: str | Path,
    config: dict[str, Any],
    audio_source_path: str | Path | None = None,
) -> bool:
    input_path = Path(input_path)
    output_path = Path(output_path)
    audio_source_path = Path(audio_source_path) if audio_source_path is not None else None
    include_audio = bool(config.get("include_audio", True)) and audio_source_path is not None and audio_source_path.exists()
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path is None:
        print("ffmpeg was not found. Keeping the OpenCV-encoded mp4 without audio; VSCode may not preview it.")
        if input_path != output_path:
            input_path.replace(output_path)
        return False

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(input_path),
    ]
    if include_audio:
        cmd.extend(["-i", str(audio_source_path), "-map", "0:v:0", "-map", "1:a?"])
    else:
        cmd.extend(["-an"])
    cmd.extend([
        "-c:v", str(config.get("ffmpeg_video_codec", "libx264")),
        "-preset", str(config.get("ffmpeg_preset", "medium")),
        "-crf", str(config.get("ffmpeg_crf", 20)),
        "-pix_fmt", "yuv420p",
    ])
    if include_audio:
        cmd.extend([
            "-c:a", str(config.get("ffmpeg_audio_codec", "aac")),
            "-b:a", str(config.get("ffmpeg_audio_bitrate", "128k")),
            "-shortest",
        ])
    cmd.extend([
        "-movflags", "+faststart",
        str(output_path),
    ])
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ffmpeg transcode failed. Keeping the OpenCV-encoded mp4; VSCode may not preview it.")
        print(result.stderr[-2000:])
        if input_path != output_path:
            input_path.replace(output_path)
        return False
    if input_path != output_path and not bool(config.get("keep_intermediate_video", False)):
        input_path.unlink(missing_ok=True)
    return True




In [18]:
# ------------------------------------------------------------
# セル概要: 描画用レイヤー
# ------------------------------------------------------------
# - グリッド線、flow 矢印、flow 数値ラベルを動画フレームに描画します。
# - 生flow動画には grid と grid_flow_arrows を常に重ねます。
# ------------------------------------------------------------

def draw_grid(frame: np.ndarray, grid_shape: tuple[int, int], config: dict[str, Any]) -> np.ndarray:
    color = tuple(config.get("line_color", (80, 220, 255)))
    thickness = int(config.get("line_thickness", 1))
    h, w = frame.shape[:2]
    gy_n, gx_n = grid_shape
    for gy in range(1, gy_n):
        y = int(h * gy / gy_n)
        cv2.line(frame, (0, y), (w - 1, y), color, thickness, lineType=cv2.LINE_AA)
    for gx in range(1, gx_n):
        x = int(w * gx / gx_n)
        cv2.line(frame, (x, 0), (x, h - 1), color, thickness, lineType=cv2.LINE_AA)
    return frame


# 関数メモ: draw_grid_flow_arrows
# - グリッドごとの平均 flow ベクトルを矢印として描きます。
# - 移動方向が見えるよう、長さの上限や最小表示量を設定で調整できます。

def draw_grid_flow_arrows(frame: np.ndarray, grid_features: list[dict[str, float]], config: dict[str, Any]) -> np.ndarray:
    color = tuple(config.get("color", (40, 255, 80)))
    tip_color = tuple(config.get("tip_color", color))
    outline_color = tuple(config.get("outline_color", (0, 0, 0)))
    thickness = int(config.get("thickness", 2))
    outline_thickness = thickness + int(config.get("outline_extra_thickness", 4))
    tip_length = float(config.get("tip_length", 0.25))
    scale = float(config.get("scale", 12.0))
    max_length_ratio = float(config.get("max_length_ratio", 0.42))
    min_magnitude = float(config.get("min_magnitude", 0.02))
    show_values = bool(config.get("show_values", True))
    value_label_color = tuple(config.get("value_label_color", (240, 245, 255)))
    value_label_outline_color = tuple(config.get("value_label_outline_color", outline_color))
    value_label_font_scale = float(config.get("value_label_font_scale", 0.42))
    value_label_thickness = int(config.get("value_label_thickness", 1))
    value_label_outline_thickness = value_label_thickness + int(config.get("value_label_outline_extra_thickness", 2))
    value_label_offset = tuple(config.get("value_label_offset", (6, 18)))
    value_label_line_gap = int(config.get("value_label_line_gap", 16))
    value_label_decimals = max(0, int(config.get("value_label_decimals", 2)))
    for row in grid_features:
        vector_mag = float(row["flow_vector_mag"])
        cx, cy = float(row["center_x"]), float(row["center_y"])
        dx, dy = float(row["flow_dx_mean"]), float(row["flow_dy_mean"])
        if show_values:
            label_x = int(np.clip(float(row["x0"]) + float(value_label_offset[0]), 0, max(frame.shape[1] - 1, 0)))
            label_y = int(np.clip(float(row["y0"]) + float(value_label_offset[1]), 0, max(frame.shape[0] - 1, 0)))
            for line_index, text in enumerate([f"x={dx:+.{value_label_decimals}f}", f"y={dy:+.{value_label_decimals}f}"]):
                org = (label_x, int(np.clip(label_y + line_index * value_label_line_gap, 0, max(frame.shape[0] - 1, 0))))
                cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, value_label_font_scale, value_label_outline_color, value_label_outline_thickness, lineType=cv2.LINE_AA)
                cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, value_label_font_scale, value_label_color, value_label_thickness, lineType=cv2.LINE_AA)
        if vector_mag < min_magnitude:
            continue
        cell_w = max(1.0, float(row["x1"] - row["x0"]))
        cell_h = max(1.0, float(row["y1"] - row["y0"]))
        max_len = min(cell_w, cell_h) * max_length_ratio
        arrow_dx = dx * scale
        arrow_dy = dy * scale
        arrow_len = float(np.hypot(arrow_dx, arrow_dy))
        if arrow_len > max_len:
            ratio = max_len / max(arrow_len, 1e-6)
            arrow_dx *= ratio
            arrow_dy *= ratio
        start = (int(round(cx - arrow_dx * 0.5)), int(round(cy - arrow_dy * 0.5)))
        end = (int(round(cx + arrow_dx * 0.5)), int(round(cy + arrow_dy * 0.5)))
        cv2.arrowedLine(frame, start, end, outline_color, outline_thickness, line_type=cv2.LINE_AA, tipLength=tip_length)
        cv2.arrowedLine(frame, start, end, color, thickness, line_type=cv2.LINE_AA, tipLength=tip_length)
        cv2.circle(frame, end, max(3, outline_thickness + 1), outline_color, -1, lineType=cv2.LINE_AA)
        cv2.circle(frame, end, max(2, thickness + 1), tip_color, -1, lineType=cv2.LINE_AA)
    return frame

# 関数メモ: render_feature_layers
# - 生flow動画用にグリッドとflow矢印をフレームへ重ねます。

def render_feature_layers(
    frame: np.ndarray,
    grid_features: list[dict[str, float]],
    config: dict[str, Any],
) -> np.ndarray:
    out = frame.copy()
    grid_shape = tuple(config["grid"]["shape"])
    out = draw_grid(out, grid_shape, config["grid"])
    out = draw_grid_flow_arrows(out, grid_features, config["flow_arrows"])
    return out


def update_display_flow_features(
    previous_features: list[dict[str, float]],
    measured_features: list[dict[str, float]],
    arrow_config: dict[str, Any],
) -> list[dict[str, float]]:
    if not bool(arrow_config.get("hold_below_min_magnitude", True)):
        return measured_features
    min_magnitude = float(arrow_config.get("min_magnitude", 0.02))
    previous_by_grid = {
        (int(row.get("grid_y", row.get("gy", 0))), int(row.get("grid_x", row.get("gx", 0)))): row
        for row in previous_features
    }
    display_features = []
    for measured in measured_features:
        grid_key = (int(measured.get("grid_y", measured.get("gy", 0))), int(measured.get("grid_x", measured.get("gx", 0))))
        previous = previous_by_grid.get(grid_key)
        measured_mag = float(measured.get("flow_vector_mag", 0.0))
        previous_mag = float(previous.get("flow_vector_mag", 0.0)) if previous is not None else 0.0
        if measured_mag >= min_magnitude or previous is None or previous_mag < min_magnitude:
            display_features.append(measured)
        else:
            held = dict(previous)
            for key in ["frame_index", "sample_index", "time_sec", "x0", "y0", "x1", "y1", "center_x", "center_y", "gy", "gx", "grid_y", "grid_x"]:
                if key in measured:
                    held[key] = measured[key]
            display_features.append(held)
    return display_features


def prepare_precomputed_grid_flow_for_render(
    precomputed_grid_flow: pd.DataFrame | None,
    frame_shape: tuple[int, ...],
    grid_shape: tuple[int, int],
    src_fps: float,
    flow_sample_sec: float,
) -> pd.DataFrame:
    if precomputed_grid_flow is None or precomputed_grid_flow.empty:
        return pd.DataFrame()
    work = precomputed_grid_flow.copy()

    time_source = "time_sec" if "time_sec" in work.columns else "time"
    time_values = pd.to_numeric(work[time_source], errors="coerce") if time_source in work.columns else pd.Series(0.0, index=work.index)
    work["time_sec"] = time_values.fillna(0.0).astype(float)

    if "grid_y" not in work.columns:
        work["grid_y"] = work.get("grid_row", work.get("gy", 0))
    if "grid_x" not in work.columns:
        work["grid_x"] = work.get("grid_col", work.get("gx", 0))
    work["grid_y"] = pd.to_numeric(work["grid_y"], errors="coerce").fillna(0).astype(int)
    work["grid_x"] = pd.to_numeric(work["grid_x"], errors="coerce").fillna(0).astype(int)
    work["gy"] = work["grid_y"]
    work["gx"] = work["grid_x"]

    if "flow_dx_mean" not in work.columns:
        work["flow_dx_mean"] = work.get("flow_x", 0.0)
    if "flow_dy_mean" not in work.columns:
        work["flow_dy_mean"] = work.get("flow_y", 0.0)
    work["flow_dx_mean"] = pd.to_numeric(work["flow_dx_mean"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    work["flow_dy_mean"] = pd.to_numeric(work["flow_dy_mean"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    if "flow_mag_mean" not in work.columns:
        work["flow_mag_mean"] = work.get("flow_magnitude_mean", np.hypot(work["flow_dx_mean"], work["flow_dy_mean"]))
    work["flow_mag_mean"] = pd.to_numeric(work["flow_mag_mean"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    if "flow_vector_mag" not in work.columns:
        work["flow_vector_mag"] = np.hypot(work["flow_dx_mean"], work["flow_dy_mean"])
    work["flow_vector_mag"] = pd.to_numeric(work["flow_vector_mag"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    if "sample_index" not in work.columns:
        work["sample_index"] = np.rint(work["time_sec"] / max(float(flow_sample_sec), 1e-6))
    if "frame_index" not in work.columns:
        work["frame_index"] = np.rint(work["time_sec"] * max(float(src_fps), 1e-6))
    work["sample_index"] = pd.to_numeric(work["sample_index"], errors="coerce").fillna(0).astype(int)
    work["frame_index"] = pd.to_numeric(work["frame_index"], errors="coerce").fillna(0).astype(int)

    cells = pd.DataFrame(flow_extract.grid_bounds(frame_shape, grid_shape)).rename(columns={"gy": "grid_y", "gx": "grid_x"})
    cells = cells[["grid_y", "grid_x", "x0", "y0", "x1", "y1"]]
    work = work.drop(columns=[c for c in ["x0", "y0", "x1", "y1", "center_x", "center_y"] if c in work.columns], errors="ignore")
    work = work.merge(cells, on=["grid_y", "grid_x"], how="inner")
    work["center_x"] = ((work["x0"] + work["x1"]) / 2).astype(float)
    work["center_y"] = ((work["y0"] + work["y1"]) / 2).astype(float)

    for col, default in {
        "valid_ratio": 1.0,
        "flow_reliable_ratio": np.nan,
        "flow_forward_backward_error_mean": np.nan,
        "flow_backward_consistency": np.nan,
        "flow_edge_confidence": 0.0,
        "flow_coherence_confidence": 1.0,
        "flow_edge_strength": 0.0,
        "flow_edge_density": 0.0,
        "flow_measurement_confidence": 0.0,
    }.items():
        if col not in work.columns:
            work[col] = default
    return work.sort_values(["sample_index", "grid_y", "grid_x"]).reset_index(drop=True)


def select_precomputed_grid_features(
    precomputed_render_grid: pd.DataFrame,
    sample_index: int,
    time_sec: float,
    flow_sample_sec: float,
) -> list[dict[str, float]]:
    if precomputed_render_grid.empty:
        return []
    part = precomputed_render_grid[precomputed_render_grid["sample_index"].eq(int(sample_index))]
    if part.empty:
        unique_times = np.asarray(sorted(precomputed_render_grid["time_sec"].dropna().unique()), dtype=float)
        if unique_times.size:
            nearest_time = float(unique_times[np.argmin(np.abs(unique_times - float(time_sec)))])
            if abs(nearest_time - float(time_sec)) <= max(float(flow_sample_sec) * 0.51, 1e-6):
                part = precomputed_render_grid[np.isclose(precomputed_render_grid["time_sec"], nearest_time)]
    if part.empty:
        return []
    return part.sort_values(["grid_y", "grid_x"]).to_dict("records")




In [19]:
# ------------------------------------------------------------
# セル概要: 可視化動画の生成処理
# ------------------------------------------------------------
# - フレーム単位flowを測定し、孤立した静止フレームを補正してから0.1秒binへ按分したgrid flowを作ります。
# - raw flow動画の矢印とグラフ出力は、cache済みまたは同じgrid flow行を参照します。CSVは出力しません。
# ------------------------------------------------------------


def create_feature_visualization_video(
    input_video_path: str | Path,
    output_video_path: str | Path,
    feature_outputs: dict[str, bool] = FEATURE_OUTPUTS,
    config: dict[str, Any] = VISUALIZATION_CONFIG,
    output_video: bool = True,
    precomputed_grid_flow: pd.DataFrame | None = None,
) -> pd.DataFrame:
    input_video_path = Path(input_video_path)
    output_video_path = Path(output_video_path)
    final_output_video_path = output_video_path
    needs_ffmpeg_postprocess = bool(
        output_video and (config.get("browser_compatible_mp4", True) or config.get("include_audio", True))
    )
    write_video_path = build_intermediate_video_path(output_video_path) if needs_ffmpeg_postprocess else output_video_path
    capture = open_video(input_video_path)
    src_fps = capture.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = 30.0
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    flow_sample_sec = max(1e-6, float(config.get("flow_sample_sec", 0.1)))
    output_fps = src_fps
    max_frames = config.get("max_frames")

    ok, first_frame = capture.read()
    if not ok:
        capture.release()
        raise RuntimeError(f"No frames were read from {input_video_path}")
    first_frame = content_update.resize_keep_aspect(first_frame, config.get("resize_width"))
    writer = make_video_writer(write_video_path, output_fps, first_frame.shape, fourcc_name=config.get("fourcc", "mp4v")) if output_video else None
    if writer is None and not feature_outputs.get("grid_flow", False):
        capture.release()
        print("raw flow video and graph feature outputs are disabled. Skipping video scan.")
        return pd.DataFrame()

    grid_shape = tuple(config["grid"]["shape"])
    precomputed_render_grid = prepare_precomputed_grid_flow_for_render(
        precomputed_grid_flow,
        first_frame.shape,
        grid_shape,
        src_fps,
        flow_sample_sec,
    )
    if len(precomputed_render_grid) > 0:
        print(f"using precomputed grid flow for rendering: {len(precomputed_render_grid)} rows")
    else:
        measured_grid_flow = flow_extract.extract_video_grid_flow(
            input_video_path,
            flow_sample_sec=flow_sample_sec,
            frame_resize_width=config.get("resize_width"),
            flow_analysis_scale=1.0,
            flow_grid=grid_shape,
            reliable_error_threshold_px=float(config.get("flow_reliable_error_threshold_px", 1.0)),
            static_frame_max_grid_vector_mag=float(config.get("flow_static_frame_max_grid_vector_mag", 0.02)),
        )
        precomputed_render_grid = prepare_precomputed_grid_flow_for_render(
            measured_grid_flow,
            first_frame.shape,
            grid_shape,
            src_fps,
            flow_sample_sec,
        )

    feature_df = precomputed_render_grid.copy() if feature_outputs.get("grid_flow", False) else pd.DataFrame()
    latest_grid_features = select_precomputed_grid_features(precomputed_render_grid, 0, 0.0, flow_sample_sec)
    if not latest_grid_features:
        latest_grid_features = flow_extract.zero_grid_flow_rows(
            first_frame.shape,
            grid_shape,
            frame_index=0,
            time_sec=0.0,
            sample_index=0,
            reliable_ratio=0.0,
            valid_ratio=0.0,
        )

    if writer is not None:
        writer.write(render_feature_layers(first_frame, latest_grid_features, config))

    written_frames = 1
    source_index = 1
    current_sample_index = 0
    progress_total = frame_count if frame_count > 0 else None
    with tqdm(total=progress_total, desc="render visualization") as pbar:
        pbar.update(1)
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if max_frames is not None and written_frames >= int(max_frames):
                break

            frame = content_update.resize_keep_aspect(frame, config.get("resize_width"))
            frame_time_sec = float(source_index / max(float(src_fps), 1e-6))
            sample_index = max(0, int(np.floor(frame_time_sec / flow_sample_sec + 1e-9)))
            if sample_index != current_sample_index:
                measured_grid_features = select_precomputed_grid_features(precomputed_render_grid, sample_index, sample_index * flow_sample_sec, flow_sample_sec)
                if measured_grid_features:
                    latest_grid_features = update_display_flow_features(latest_grid_features, measured_grid_features, config["flow_arrows"])
                    current_sample_index = sample_index

            if writer is not None:
                writer.write(render_feature_layers(frame, latest_grid_features, config))
            source_index += 1
            written_frames += 1
            pbar.update(1)

    capture.release()
    if writer is not None:
        writer.release()
        if needs_ffmpeg_postprocess:
            audio_source_path = find_visualization_audio_source(input_video_path) if config.get("include_audio", True) else None
            transcoded = transcode_to_browser_compatible_mp4(
                write_video_path,
                final_output_video_path,
                config,
                audio_source_path=audio_source_path,
            )
            audio_label = " + audio" if audio_source_path is not None and Path(audio_source_path).exists() else ""
            codec_label = f"H.264/yuv420p{audio_label}" if transcoded else str(config.get("fourcc", "mp4v"))
        else:
            codec_label = str(config.get("fourcc", "mp4v"))
        print(f"saved video   : {final_output_video_path} ({codec_label})")
    else:
        print("raw flow video output skipped.")

    print(f"frame-to-frame flow with isolated static-frame split apportioned into {flow_sample_sec:.3f} sec bins; output fps: {output_fps:.2f}")
    print(f"written frames: {written_frames}")
    return feature_df


In [20]:
# ------------------------------------------------------------
# セル概要: 指定18項目の出力生成
# ------------------------------------------------------------
# - flow / 信頼度重み付きflowの3x3グラフと広域振動スコアだけを出力します。
# - 3x3グラフには、同じ系列から算出した振動スコアを背景として常に重ねます。
# ------------------------------------------------------------

import matplotlib.pyplot as plt


PLOT_METRIC_SPECS = {
    "vector_change": {"label": "vector_change", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:purple"},
    "flow_x": {"label": "flow_x", "nonnegative": False, "score_family": "oscillation", "overlay_color": "tab:orange"},
    "flow_y": {"label": "flow_y", "nonnegative": False, "score_family": "oscillation", "overlay_color": "tab:orange"},
    "t_vector_change": {"label": "t_vector_change", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:purple"},
    "t_flow_x": {"label": "t_flow_x", "nonnegative": False, "score_family": "oscillation", "overlay_color": "tab:orange"},
    "t_flow_y": {"label": "t_flow_y", "nonnegative": False, "score_family": "oscillation", "overlay_color": "tab:orange"},
    "flow_edge_confidence": {"label": "edge confidence", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:green", "fixed_ylim": (-0.05, 1.05)},
    "flow_coherence_confidence": {"label": "flow coherence confidence", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:cyan", "fixed_ylim": (-0.05, 1.05)},
    "flow_backward_consistency": {"label": "forward-backward consistency", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:brown", "fixed_ylim": (-0.05, 1.05)},
    "flow_measurement_confidence": {"label": "measurement confidence", "nonnegative": True, "score_family": "sustained_change", "overlay_color": "tab:olive", "fixed_ylim": (-0.05, 1.05)},
}
FLOW_ACCEL_OVERLAY_SPECS = {
    "flow_x": {"column": "flow_x_accel_0p1s", "label": "d_flow_x / 0.1s", "color": "tab:red"},
    "flow_y": {"column": "flow_y_accel_0p1s", "label": "d_flow_y / 0.1s", "color": "tab:red"},
    "t_flow_x": {"column": "t_flow_x_accel_0p1s", "label": "d_t_flow_x / 0.1s", "color": "tab:red"},
    "t_flow_y": {"column": "t_flow_y_accel_0p1s", "label": "d_t_flow_y / 0.1s", "color": "tab:red"},
}
SUSTAINED_CHANGE_SCORE_WEIGHTS = {
    "sustained_sum": 0.35,
    "sustained_p95": 0.25,
    "sustained_high_ratio": 0.25,
    "sustained_top_mean": 0.15,
}
OSCILLATION_SCORE_WEIGHTS = {
    "osc_range_p90_p10": 0.20,
    "osc_residual_mean": 0.20,
    "osc_diff_sum": 0.15,
    "osc_balanced_diff_sum": 0.30,
    "osc_turn_ratio": 0.15,
}
BROAD_OUTPUT_TO_METRIC = {
    "vector_change_broad_vib_score": "vector_change",
    "flow_x_broad_vib_score": "flow_x",
    "flow_y_broad_vib_score": "flow_y",
    "t_vector_change_broad_vib_score": "t_vector_change",
    "t_flow_x_broad_vib_score": "t_flow_x",
    "t_flow_y_broad_vib_score": "t_flow_y",
}
MOTION_IMPULSE_OUTPUT_KEYS = {"motion_impulse_score"}
ACCEL_IMPACT_OUTPUT_SPECS = {
    "accel_impact_x_score": {"axis": "x", "score_column": "accel_impact_x_score", "signed_column": "signed_accel_impact_x_score", "label": "accel impact x"},
    "accel_impact_y_score": {"axis": "y", "score_column": "accel_impact_y_score", "signed_column": "signed_accel_impact_y_score", "label": "accel impact y"},
    "t_accel_impact_x_score": {"axis": "t_x", "score_column": "t_accel_impact_x_score", "signed_column": "signed_t_accel_impact_x_score", "label": "t accel impact x"},
    "t_accel_impact_y_score": {"axis": "t_y", "score_column": "t_accel_impact_y_score", "signed_column": "signed_t_accel_impact_y_score", "label": "t accel impact y"},
}
GRAPH_OUTPUT_KEYS = set(PLOT_METRIC_SPECS) | set(BROAD_OUTPUT_TO_METRIC) | MOTION_IMPULSE_OUTPUT_KEYS | set(ACCEL_IMPACT_OUTPUT_SPECS)


def output_enabled(key: str, settings: dict[str, bool] | None = None) -> bool:
    active_settings = OUTPUT_SETTINGS if settings is None else settings
    return bool(active_settings.get(key, False))


def any_graph_output_enabled(settings: dict[str, bool] | None = None) -> bool:
    settings = settings or OUTPUT_SETTINGS
    return any(bool(settings.get(key, False)) for key in GRAPH_OUTPUT_KEYS)


def enabled_metric_keys(settings: dict[str, bool] | None = None) -> list[str]:
    settings = settings or OUTPUT_SETTINGS
    keys = {key for key in PLOT_METRIC_SPECS if output_enabled(key, settings)}
    keys.update(metric for broad_key, metric in BROAD_OUTPUT_TO_METRIC.items() if output_enabled(broad_key, settings))
    return [key for key in PLOT_METRIC_SPECS if key in keys]


def normalize_grid_flow_features(features_df: pd.DataFrame) -> pd.DataFrame:
    if features_df is None or features_df.empty:
        return pd.DataFrame()
    work = features_df.copy()
    work["time"] = work.get("time", work.get("time_sec", 0.0))
    work["grid_row"] = work.get("grid_row", work.get("grid_y", work.get("gy", 0)))
    work["grid_col"] = work.get("grid_col", work.get("grid_x", work.get("gx", 0)))
    work["flow_x"] = work.get("flow_x", work.get("flow_dx_mean", 0.0))
    work["flow_y"] = work.get("flow_y", work.get("flow_dy_mean", 0.0))
    if "flow_reliable_ratio" not in work.columns:
        work["flow_reliable_ratio"] = 1.0
    if "flow_backward_consistency" not in work.columns:
        work["flow_backward_consistency"] = work["flow_reliable_ratio"]
    else:
        work["flow_backward_consistency"] = work["flow_backward_consistency"].fillna(work["flow_reliable_ratio"])
    if "flow_edge_confidence" not in work.columns:
        work["flow_edge_confidence"] = 0.0
    if "flow_coherence_confidence" not in work.columns:
        work["flow_coherence_confidence"] = 1.0
    if "flow_measurement_confidence" not in work.columns:
        work["flow_measurement_confidence"] = 0.0
    if "sample_id" not in work.columns:
        work["sample_id"] = "unknown"
    if "view" not in work.columns:
        work["view"] = "front"

    keep_cols = [
        "sample_id",
        "view",
        "time",
        "grid_row",
        "grid_col",
        "flow_x",
        "flow_y",
        "flow_reliable_ratio",
        "flow_backward_consistency",
        "flow_edge_confidence",
        "flow_coherence_confidence",
        "flow_measurement_confidence",
    ]
    out = work[keep_cols].copy()
    out["sample_id"] = out["sample_id"].astype(str)
    out["view"] = out["view"].astype(str).str.lower().str.strip()
    out["time"] = pd.to_numeric(out["time"], errors="coerce").fillna(0.0).astype(float)
    out["grid_row"] = pd.to_numeric(out["grid_row"], errors="coerce").fillna(0).astype(int)
    out["grid_col"] = pd.to_numeric(out["grid_col"], errors="coerce").fillna(0).astype(int)
    out["flow_x"] = pd.to_numeric(out["flow_x"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    out["flow_y"] = pd.to_numeric(out["flow_y"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    out["flow_reliable_ratio"] = pd.to_numeric(out["flow_reliable_ratio"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0).astype(float)
    out["flow_backward_consistency"] = pd.to_numeric(out["flow_backward_consistency"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0).astype(float)
    out["flow_edge_confidence"] = pd.to_numeric(out["flow_edge_confidence"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0).astype(float)
    out["flow_coherence_confidence"] = pd.to_numeric(out["flow_coherence_confidence"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.0, 1.0).astype(float)
    out["flow_measurement_confidence"] = pd.to_numeric(out["flow_measurement_confidence"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0).astype(float)
    return out.sort_values(["sample_id", "view", "grid_row", "grid_col", "time"]).reset_index(drop=True)


def infer_grid_shape(grid_df: pd.DataFrame) -> tuple[int, int]:
    if grid_df.empty:
        return (3, 3)
    return (
        int(pd.to_numeric(grid_df["grid_row"], errors="coerce").max()) + 1,
        int(pd.to_numeric(grid_df["grid_col"], errors="coerce").max()) + 1,
    )


def common_ylim(values: list[Any], *, nonnegative: bool) -> tuple[float, float]:
    finite_values: list[float] = [0.0]
    for value in values:
        arr = np.asarray(value, dtype=float).reshape(-1)
        arr = arr[np.isfinite(arr)]
        if arr.size:
            finite_values.extend(arr.tolist())
    ymin, ymax = float(np.min(finite_values)), float(np.max(finite_values))
    pad = max((ymax - ymin) * 0.05, max(abs(ymax), abs(ymin)) * 0.02, 1e-6)
    ymin, ymax = ymin - pad, ymax + pad
    if nonnegative:
        ymin = 0.0
    if ymin >= ymax:
        ymax = ymin + 1.0
    return ymin, ymax


def add_plot_metrics(grid_df: pd.DataFrame) -> pd.DataFrame:
    if grid_df.empty:
        return pd.DataFrame()
    parts = []
    for _, group in grid_df.groupby(["sample_id", "view", "grid_row", "grid_col"], sort=False):
        part = group.sort_values("time").copy()
        times = part["time"].to_numpy(dtype=float)
        x = part["flow_x"].to_numpy(dtype=float)
        y = part["flow_y"].to_numpy(dtype=float)
        flow_sample_sec = max(float(VISUALIZATION_CONFIG.get("flow_sample_sec", 0.1)), 1e-6)
        prev_times = np.r_[times[:1], times[:-1]] if times.size else times
        dt = times - prev_times
        dt = np.where(dt > 1e-9, dt, flow_sample_sec)
        prev_x = np.r_[x[:1], x[:-1]] if x.size else x
        prev_y = np.r_[y[:1], y[:-1]] if y.size else y
        reliable = part["flow_reliable_ratio"].to_numpy(dtype=float)
        reliable_low = float(VISUALIZATION_CONFIG.get("t_flow_reliable_weight_low", 0.35))
        reliable_high = float(VISUALIZATION_CONFIG.get("t_flow_reliable_weight_high", 0.85))
        reliable_gamma = max(float(VISUALIZATION_CONFIG.get("t_flow_reliable_weight_gamma", 2.0)), 1e-6)
        reliable_weight = flow_tensor.reliable_ratio_soft_gate(reliable, low=reliable_low, high=reliable_high, gamma=reliable_gamma)
        delta_x = x - prev_x
        delta_y = y - prev_y
        part["delta_flow_x"] = delta_x
        part["delta_flow_y"] = delta_y
        part["flow_x_accel_0p1s"] = (delta_x / dt * flow_sample_sec).astype(float)
        part["flow_y_accel_0p1s"] = (delta_y / dt * flow_sample_sec).astype(float)
        part["vector_change"] = np.hypot(delta_x, delta_y)
        part["t_vector_change"] = part["vector_change"].to_numpy(dtype=float) * reliable_weight
        t_flow_x = x * reliable_weight
        t_flow_y = y * reliable_weight
        prev_t_flow_x = np.r_[t_flow_x[:1], t_flow_x[:-1]] if t_flow_x.size else t_flow_x
        prev_t_flow_y = np.r_[t_flow_y[:1], t_flow_y[:-1]] if t_flow_y.size else t_flow_y
        part["t_flow_x"] = t_flow_x
        part["t_flow_y"] = t_flow_y
        part["t_flow_x_accel_0p1s"] = ((t_flow_x - prev_t_flow_x) / dt * flow_sample_sec).astype(float)
        part["t_flow_y_accel_0p1s"] = ((t_flow_y - prev_t_flow_y) / dt * flow_sample_sec).astype(float)
        for accel_col in ["flow_x_accel_0p1s", "flow_y_accel_0p1s", "t_flow_x_accel_0p1s", "t_flow_y_accel_0p1s"]:
            if len(part):
                part.iloc[0, part.columns.get_loc(accel_col)] = 0.0
        parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def significant_turn_ratio(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if values.size < 3:
        return 0.0
    diffs = np.diff(values)
    finite_diffs = diffs[np.isfinite(diffs)]
    if finite_diffs.size < 2:
        return 0.0
    threshold = max(float(np.percentile(np.abs(finite_diffs), 50)) * 0.25, 1e-9)
    significant = np.abs(diffs) >= threshold
    turn_mask = significant[1:] & significant[:-1] & (diffs[1:] * diffs[:-1] < 0.0)
    denominator = max(1, int(np.count_nonzero(significant)) - 1)
    return float(np.count_nonzero(turn_mask) / denominator)


def significant_direction_balance(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        return 0.0
    diffs = np.diff(values)
    finite_diffs = diffs[np.isfinite(diffs)]
    if finite_diffs.size == 0:
        return 0.0
    threshold = max(float(np.percentile(np.abs(finite_diffs), 50)) * 0.25, 1e-9)
    positive_sum = float(np.sum(finite_diffs[finite_diffs >= threshold]))
    negative_sum = float(np.sum(np.abs(finite_diffs[finite_diffs <= -threshold])))
    denominator = max(positive_sum, negative_sum)
    return float(min(positive_sum, negative_sum) / denominator) if denominator > 1e-12 else 0.0


def build_sustained_change_window_features(segment: np.ndarray, high_ratio_fraction: float) -> dict[str, float]:
    values = np.clip(np.nan_to_num(np.asarray(segment, dtype=float), nan=0.0, posinf=0.0, neginf=0.0), 0.0, None)
    if values.size == 0:
        values = np.zeros(1, dtype=float)
    change_max = float(np.max(values))
    top_count = max(1, int(np.ceil(values.size * 0.30)))
    top_mean = float(np.mean(np.sort(values)[-top_count:])) if values.size else 0.0
    high_ratio = float(np.mean(values >= change_max * high_ratio_fraction)) if change_max > 1e-6 else 0.0
    return {
        "score_family": "sustained_change",
        "change_mean": float(np.mean(values)),
        "change_sum": float(np.sum(values)),
        "change_p95": float(np.percentile(values, 95)),
        "change_max": change_max,
        "change_std": float(np.std(values)),
        "change_variation": float(np.mean(np.abs(np.diff(values)))) if values.size >= 2 else 0.0,
        "change_high_ratio": high_ratio,
        "sustained_sum": float(np.sum(values)),
        "sustained_p95": float(np.percentile(values, 95)),
        "sustained_high_ratio": high_ratio,
        "sustained_top_mean": top_mean,
    }


def build_oscillation_window_features(segment: np.ndarray, high_ratio_fraction: float) -> dict[str, float]:
    values = np.nan_to_num(np.asarray(segment, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    if values.size == 0:
        values = np.zeros(1, dtype=float)
    center = float(np.median(values))
    residual_abs = np.abs(values - center)
    signed_diffs = np.diff(values) if values.size >= 2 else np.zeros(0, dtype=float)
    diffs = np.abs(signed_diffs)
    diff_max = float(np.max(diffs)) if diffs.size else 0.0
    diff_high_ratio = float(np.mean(diffs >= diff_max * high_ratio_fraction)) if diff_max > 1e-6 and diffs.size else 0.0
    range_p90_p10 = float(np.percentile(values, 90) - np.percentile(values, 10)) if values.size else 0.0
    residual_mean = float(np.mean(residual_abs)) if residual_abs.size else 0.0
    diff_sum = float(np.sum(diffs)) if diffs.size else 0.0
    diff_p95 = float(np.percentile(diffs, 95)) if diffs.size else 0.0
    direction_balance = significant_direction_balance(values)
    turn_ratio = significant_turn_ratio(values)
    return {
        "score_family": "oscillation",
        "change_mean": residual_mean,
        "change_sum": float(np.sum(residual_abs) + diff_sum),
        "change_p95": max(float(np.percentile(residual_abs, 95)) if residual_abs.size else 0.0, diff_p95),
        "change_max": max(float(np.max(residual_abs)) if residual_abs.size else 0.0, diff_max),
        "change_std": float(np.std(values)),
        "change_variation": diff_sum / max(int(diffs.size), 1),
        "change_high_ratio": diff_high_ratio,
        "osc_range_p90_p10": range_p90_p10,
        "osc_residual_mean": residual_mean,
        "osc_diff_sum": diff_sum,
        "osc_direction_balance": direction_balance,
        "osc_balanced_diff_sum": diff_sum * direction_balance,
        "osc_diff_high_ratio": diff_high_ratio,
        "osc_turn_ratio": turn_ratio,
    }


def build_metric_window_features(metric_key: str, segment: np.ndarray, high_ratio_fraction: float) -> dict[str, float]:
    family = str(PLOT_METRIC_SPECS[metric_key]["score_family"])
    if family == "sustained_change":
        return build_sustained_change_window_features(segment, high_ratio_fraction)
    if family == "oscillation":
        return build_oscillation_window_features(segment, high_ratio_fraction)
    raise ValueError(f"Unknown score family: {family}")


def add_vibration_score_columns(window_df: pd.DataFrame) -> pd.DataFrame:
    work = window_df.copy()
    family = str(work["score_family"].iloc[0]) if "score_family" in work.columns and len(work) else "sustained_change"
    weights = SUSTAINED_CHANGE_SCORE_WEIGHTS if family == "sustained_change" else OSCILLATION_SCORE_WEIGHTS
    for name in weights:
        if name not in work.columns:
            work[name] = 0.0
        work[f"{name}_z"] = feature_vibration.robust_positive_zscore(work[name])
    work["vibration_score_raw"] = sum(weight * work[f"{name}_z"] for name, weight in weights.items()).astype(float)
    work["vibration_score"] = feature_vibration.percentile_normalize_0_1(
        work["vibration_score_raw"],
        lower_percentile=0.0,
        upper_percentile=95.0,
        min_visible=1e-6,
    )
    return feature_vibration.add_vibration_score_alpha(
        work,
        alpha_min=float(VIBRATION_SCORE_CONFIG["alpha_min"]),
        alpha_max=float(VIBRATION_SCORE_CONFIG["alpha_max"]),
        min_visible=1e-6,
    )


def build_vibration_scores(metric_df: pd.DataFrame, metric_keys: list[str]) -> pd.DataFrame:
    rows = []
    window_sec = float(VIBRATION_SCORE_CONFIG["window_sec"])
    hop_sec = float(VIBRATION_SCORE_CONFIG["hop_sec"])
    high_ratio_fraction = float(VIBRATION_SCORE_CONFIG["high_ratio_fraction"])
    for _, group in metric_df.groupby(["sample_id", "view", "grid_row", "grid_col"], sort=False):
        part = group.sort_values("time")
        times = part["time"].to_numpy(dtype=float)
        if times.size == 0:
            continue
        duration_sec = max(float(np.nanmax(times)) if np.isfinite(times).any() else 0.0, window_sec)
        for metric_key in metric_keys:
            metric_values = part[metric_key].to_numpy(dtype=float)
            feature_rows = []
            for start_sec in feature_vibration.build_flow_window_starts(duration_sec, window_sec, hop_sec):
                end_sec = float(min(start_sec + window_sec, duration_sec))
                start_sec = max(end_sec - window_sec, 0.0)
                mask = (times >= start_sec) & (times < end_sec)
                if not np.any(mask):
                    mask = np.zeros_like(times, dtype=bool)
                    mask[int(np.argmin(np.abs(times - start_sec)))] = True
                segment = metric_values[mask]
                feature_rows.append({
                    "sample_id": str(part["sample_id"].iat[0]),
                    "view": str(part["view"].iat[0]),
                    "grid_row": int(part["grid_row"].iat[0]),
                    "grid_col": int(part["grid_col"].iat[0]),
                    "metric_key": metric_key,
                    "window_start_bin": int(round(start_sec / max(hop_sec, 1e-6))),
                    "window_start_sec": float(start_sec),
                    "window_end_sec": float(end_sec),
                    "window_center_sec": float(0.5 * (start_sec + end_sec)),
                    **build_metric_window_features(metric_key, segment, high_ratio_fraction),
                })
            if feature_rows:
                rows.append(add_vibration_score_columns(pd.DataFrame(feature_rows)))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def build_broad_vib_scores(vibration_df: pd.DataFrame) -> pd.DataFrame:
    if vibration_df.empty:
        return pd.DataFrame()
    rows = []
    broad_weights = VIBRATION_SCORE_CONFIG.get("broad_vib_score_weights", {}) or {}
    weight_a = float(broad_weights.get("A", broad_weights.get("a", 1.0)))
    weight_b = float(broad_weights.get("B", broad_weights.get("b", 0.0)))
    weight_c = float(broad_weights.get("C", broad_weights.get("c", 0.0)))
    low_intensity_percentile = float(np.clip(VIBRATION_SCORE_CONFIG.get("broad_vib_low_intensity_percentile", 20.0), 0.0, 100.0))
    spread_power = max(float(VIBRATION_SCORE_CONFIG.get("broad_vib_spread_power", 1.0)), 0.0)
    for keys, group in vibration_df.groupby(["sample_id", "view", "metric_key", "window_start_bin"], sort=False):
        sample_id, view, metric_key, window_start_bin = keys
        scores = pd.to_numeric(group["vibration_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0).to_numpy(dtype=float)
        change_sums = pd.to_numeric(group["change_sum"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0.0).to_numpy(dtype=float)
        vibration_intensities = scores * change_sums
        mean_vibration_intensity = float(np.mean(vibration_intensities)) if vibration_intensities.size else 0.0
        low_percentile_vibration_intensity = float(np.percentile(vibration_intensities, low_intensity_percentile)) if vibration_intensities.size else 0.0
        max_vibration_intensity = float(np.max(vibration_intensities)) if vibration_intensities.size else 0.0
        value_sum = float(np.sum(vibration_intensities)) if vibration_intensities.size else 0.0
        value_square_sum = float(np.sum(vibration_intensities * vibration_intensities)) if vibration_intensities.size else 0.0
        effective_cells = (value_sum * value_sum / value_square_sum) if value_square_sum > 1e-12 else 0.0
        grid_count = int(len(group))
        spread_score = float(np.clip(effective_cells / grid_count, 0.0, 1.0)) if grid_count else 0.0
        spread_score_weighted = float(np.power(spread_score, spread_power))
        broad_vib_score = (
            weight_a * mean_vibration_intensity * spread_score_weighted
            + weight_b * low_percentile_vibration_intensity
            + weight_c * max_vibration_intensity
        )
        rows.append({
            "sample_id": str(sample_id),
            "view": str(view),
            "metric_key": str(metric_key),
            "broad_output_key": next((key for key, value in BROAD_OUTPUT_TO_METRIC.items() if value == metric_key), str(metric_key)),
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "broad_vib_score": float(broad_vib_score),
            "mean_vibration_score": float(np.mean(scores)) if scores.size else 0.0,
            "mean_change_sum": float(np.mean(change_sums)) if change_sums.size else 0.0,
            "max_change_sum": float(np.max(change_sums)) if change_sums.size else 0.0,
            "mean_vibration_intensity": mean_vibration_intensity,
            "low_percentile_vibration_intensity": low_percentile_vibration_intensity,
            "low_intensity_percentile": low_intensity_percentile,
            "max_vibration_intensity": max_vibration_intensity,
            "spread_score": spread_score,
            "spread_score_weighted": spread_score_weighted,
            "spread_power": spread_power,
            "effective_cells": float(effective_cells),
            "grid_count": grid_count,
            "broad_weight_a": weight_a,
            "broad_weight_b": weight_b,
            "broad_weight_c": weight_c,
        })
    return pd.DataFrame(rows).sort_values(["sample_id", "view", "metric_key", "window_start_sec"]).reset_index(drop=True)


def signed_accel_impact_components(values: np.ndarray, grid_count: int, eps: float) -> dict[str, float]:
    values = np.nan_to_num(np.asarray(values, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    grid_count = max(1, int(grid_count))
    pos_sum = float(np.sum(np.clip(values, 0.0, None)))
    neg_sum = float(np.sum(np.clip(-values, 0.0, None)))
    major = max(pos_sum, neg_sum)
    minor = min(pos_sum, neg_sum)
    mass = float(np.clip(major / grid_count, 0.0, 1.0))
    dominance = float((major - minor) / max(major + minor + float(eps), float(eps)))
    score = float(np.clip(mass * dominance, 0.0, 1.0))
    direction = 1.0 if pos_sum > neg_sum else -1.0 if neg_sum > pos_sum else 0.0
    return {
        "score": score,
        "signed_score": score * direction,
        "direction": direction,
        "mass": mass,
        "dominance": dominance,
        "pos_sum": pos_sum,
        "neg_sum": neg_sum,
        "major_sum": major,
        "minor_sum": minor,
    }


def build_accel_impact_scores(metric_df: pd.DataFrame) -> pd.DataFrame:
    if metric_df.empty:
        return pd.DataFrame()
    accel_specs = {
        "accel_impact_x": ("flow_x_accel_0p1s", max(float(ACCEL_IMPACT_SCORE_CONFIG.get("x_clip_abs", 1.0)), 1e-6)),
        "accel_impact_y": ("flow_y_accel_0p1s", max(float(ACCEL_IMPACT_SCORE_CONFIG.get("y_clip_abs", 1.0)), 1e-6)),
        "t_accel_impact_x": ("t_flow_x_accel_0p1s", max(float(ACCEL_IMPACT_SCORE_CONFIG.get("t_x_clip_abs", 1.0)), 1e-6)),
        "t_accel_impact_y": ("t_flow_y_accel_0p1s", max(float(ACCEL_IMPACT_SCORE_CONFIG.get("t_y_clip_abs", 1.0)), 1e-6)),
    }
    required = {"sample_id", "view", "time", "grid_row", "grid_col", *(column for column, _ in accel_specs.values())}
    if required - set(metric_df.columns):
        return pd.DataFrame()
    eps = max(float(ACCEL_IMPACT_SCORE_CONFIG.get("eps", 1e-6)), 1e-12)
    window_sec = float(VIBRATION_SCORE_CONFIG["window_sec"])
    hop_sec = float(VIBRATION_SCORE_CONFIG["hop_sec"])
    rows = []
    for (sample_id, view), group in metric_df.groupby(["sample_id", "view"], sort=False):
        work = group.sort_values(["time", "grid_row", "grid_col"]).copy()
        grid_count_default = max(1, int(work[["grid_row", "grid_col"]].drop_duplicates().shape[0]))
        time_rows = []
        for time_value, time_group in work.groupby("time", sort=True):
            grid_count = max(grid_count_default, int(len(time_group)))
            time_row = {"time": float(time_value), "accel_impact_grid_count": int(grid_count)}
            for prefix, (column, clip_abs) in accel_specs.items():
                accel = pd.to_numeric(time_group[column], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
                normalized = np.clip(accel / clip_abs, -1.0, 1.0)
                comp = signed_accel_impact_components(normalized, grid_count, eps)
                time_row[f"{prefix}_score"] = float(comp["score"])
                time_row[f"signed_{prefix}_score"] = float(comp["signed_score"])
                time_row[f"{prefix}_direction"] = float(comp["direction"])
                time_row[f"{prefix}_mass"] = float(comp["mass"])
                time_row[f"{prefix}_dominance"] = float(comp["dominance"])
                time_row[f"{prefix}_pos_sum"] = float(comp["pos_sum"])
                time_row[f"{prefix}_neg_sum"] = float(comp["neg_sum"])
                time_row[f"{prefix}_clip_abs"] = float(clip_abs)
            time_rows.append(time_row)
        time_df = pd.DataFrame(time_rows).sort_values("time").reset_index(drop=True) if time_rows else pd.DataFrame()
        if time_df.empty:
            continue
        times = time_df["time"].to_numpy(dtype=float)
        duration_sec = max(float(np.nanmax(times)) if np.isfinite(times).any() else 0.0, window_sec)
        for start_sec in feature_vibration.build_flow_window_starts(duration_sec, window_sec, hop_sec):
            end_sec = float(min(start_sec + window_sec, duration_sec))
            start_sec = max(end_sec - window_sec, 0.0)
            mask = (times >= start_sec) & (times < end_sec)
            if not np.any(mask):
                mask = np.zeros_like(times, dtype=bool)
                mask[int(np.argmin(np.abs(times - start_sec)))] = True
            window_part = time_df.loc[mask].copy()
            row = {
                "sample_id": str(sample_id),
                "view": str(view),
                "window_start_bin": int(round(start_sec / max(hop_sec, 1e-6))),
                "window_start_sec": float(start_sec),
                "window_end_sec": float(end_sec),
                "window_center_sec": float(0.5 * (start_sec + end_sec)),
                "time": float(0.5 * (start_sec + end_sec)),
                "accel_impact_grid_count": int(grid_count_default),
                "accel_impact_sample_count": int(len(window_part)),
            }
            for prefix, (_, clip_abs) in accel_specs.items():
                row[f"{prefix}_score"] = float(window_part[f"{prefix}_score"].mean())
                row[f"signed_{prefix}_score"] = float(window_part[f"signed_{prefix}_score"].mean())
                row[f"{prefix}_mass"] = float(window_part[f"{prefix}_mass"].mean())
                row[f"{prefix}_dominance"] = float(window_part[f"{prefix}_dominance"].mean())
                row[f"{prefix}_pos_sum"] = float(window_part[f"{prefix}_pos_sum"].mean())
                row[f"{prefix}_neg_sum"] = float(window_part[f"{prefix}_neg_sum"].mean())
                row[f"{prefix}_clip_abs"] = float(clip_abs)
            rows.append(row)
    return pd.DataFrame(rows).sort_values(["sample_id", "view", "window_start_sec"]).reset_index(drop=True) if rows else pd.DataFrame()


def build_motion_impulse_scores(metric_df: pd.DataFrame) -> pd.DataFrame:
    if metric_df.empty:
        return pd.DataFrame()
    required = {"sample_id", "view", "time", "grid_row", "grid_col", "vector_change", "delta_flow_x", "delta_flow_y"}
    if required - set(metric_df.columns):
        return pd.DataFrame()

    intensity_percentile = float(np.clip(MOTION_IMPULSE_SCORE_CONFIG.get("intensity_percentile", 75.0), 0.0, 100.0))
    intensity_upper_percentile = float(np.clip(MOTION_IMPULSE_SCORE_CONFIG.get("intensity_upper_percentile", 95.0), 0.0, 100.0))
    min_vector_change = max(float(MOTION_IMPULSE_SCORE_CONFIG.get("min_vector_change", 1e-6)), 0.0)
    spread_power = max(float(MOTION_IMPULSE_SCORE_CONFIG.get("spread_power", 1.0)), 0.0)
    direction_power = max(float(MOTION_IMPULSE_SCORE_CONFIG.get("direction_power", 0.75)), 0.0)
    intensity_gate_power = max(float(MOTION_IMPULSE_SCORE_CONFIG.get("intensity_gate_power", 0.5)), 0.0)
    component_weights = MOTION_IMPULSE_SCORE_CONFIG.get("component_weights", {}) or {}
    intensity_weight = max(float(component_weights.get("intensity", 0.15)), 0.0)
    spread_weight = max(float(component_weights.get("spread", 0.45)), 0.0)
    direction_weight = max(float(component_weights.get("direction", 0.40)), 0.0)
    component_weight_sum = max(intensity_weight + spread_weight + direction_weight, 1e-6)
    rows = []

    for (sample_id, view), group in metric_df.groupby(["sample_id", "view"], sort=False):
        work = group.sort_values(["time", "grid_row", "grid_col"]).copy()
        raw_change = pd.to_numeric(work["vector_change"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0.0).to_numpy(dtype=float)
        change_strength = feature_vibration.robust_positive_zscore(raw_change)
        change_strength = np.where(raw_change > min_vector_change, change_strength, 0.0).astype(float)
        work["_motion_impulse_change_strength"] = change_strength
        grid_count_default = max(1, int(work[["grid_row", "grid_col"]].drop_duplicates().shape[0]))
        time_rows = []

        for time_value, time_group in work.groupby("time", sort=True):
            strengths = time_group["_motion_impulse_change_strength"].to_numpy(dtype=float)
            dx = pd.to_numeric(time_group["delta_flow_x"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
            dy = pd.to_numeric(time_group["delta_flow_y"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
            changes = pd.to_numeric(time_group["vector_change"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0.0).to_numpy(dtype=float)
            magnitudes = np.hypot(dx, dy)
            valid = (strengths > 0.0) & (magnitudes > min_vector_change)
            grid_count = max(grid_count_default, int(len(time_group)))
            total_strength = float(np.sum(strengths[valid])) if np.any(valid) else 0.0

            if total_strength > 1e-12:
                valid_strengths = strengths[valid]
                unit_x = dx[valid] / np.maximum(magnitudes[valid], min_vector_change)
                unit_y = dy[valid] / np.maximum(magnitudes[valid], min_vector_change)
                direction_coherence = float(np.hypot(np.sum(valid_strengths * unit_x), np.sum(valid_strengths * unit_y)) / total_strength)
                strength_square_sum = float(np.sum(valid_strengths * valid_strengths))
                effective_cells = float((total_strength * total_strength) / strength_square_sum) if strength_square_sum > 1e-12 else 0.0
                intensity_raw = float(np.percentile(valid_strengths, intensity_percentile))
                active_grid_ratio = float(np.count_nonzero(valid) / grid_count)
            else:
                direction_coherence = 0.0
                effective_cells = 0.0
                intensity_raw = 0.0
                active_grid_ratio = 0.0

            time_rows.append({
                "sample_id": str(sample_id),
                "view": str(view),
                "time": float(time_value),
                "motion_impulse_intensity_raw": intensity_raw,
                "motion_impulse_spread_score": float(np.clip(effective_cells / grid_count, 0.0, 1.0)),
                "motion_impulse_direction_coherence": float(np.clip(direction_coherence, 0.0, 1.0)),
                "motion_impulse_effective_cells": float(effective_cells),
                "motion_impulse_grid_count": int(grid_count),
                "motion_impulse_active_grid_ratio": active_grid_ratio,
                "motion_impulse_vector_change_sum": float(np.sum(changes)) if changes.size else 0.0,
                "motion_impulse_vector_change_p95": float(np.percentile(changes, 95)) if changes.size else 0.0,
                "motion_impulse_intensity_percentile": intensity_percentile,
                "motion_impulse_spread_power": spread_power,
                "motion_impulse_direction_power": direction_power,
                "motion_impulse_intensity_gate_power": intensity_gate_power,
                "motion_impulse_intensity_weight": intensity_weight,
                "motion_impulse_spread_weight": spread_weight,
                "motion_impulse_direction_weight": direction_weight,
            })

        view_scores = pd.DataFrame(time_rows)
        if view_scores.empty:
            continue
        view_scores["motion_impulse_intensity_score"] = feature_vibration.percentile_normalize_0_1(
            view_scores["motion_impulse_intensity_raw"],
            lower_percentile=0.0,
            upper_percentile=intensity_upper_percentile,
            min_visible=1e-6,
        )
        spread = view_scores["motion_impulse_spread_score"].to_numpy(dtype=float)
        direction = view_scores["motion_impulse_direction_coherence"].to_numpy(dtype=float)
        intensity = view_scores["motion_impulse_intensity_score"].to_numpy(dtype=float)
        spread_component = np.power(np.clip(spread, 0.0, 1.0), spread_power)
        direction_component = np.power(np.clip(direction, 0.0, 1.0), direction_power)
        intensity_gate = np.power(np.clip(intensity, 0.0, 1.0), intensity_gate_power)
        view_scores["motion_impulse_spread_component"] = spread_component.astype(float)
        view_scores["motion_impulse_direction_component"] = direction_component.astype(float)
        view_scores["motion_impulse_intensity_gate"] = intensity_gate.astype(float)
        view_scores["motion_impulse_component_score"] = (
            intensity_weight * intensity
            + spread_weight * spread_component
            + direction_weight * direction_component
        ) / component_weight_sum
        view_scores["motion_impulse_score_raw"] = (
            intensity_gate
            * view_scores["motion_impulse_component_score"].to_numpy(dtype=float)
        ).astype(float)
        view_scores["motion_impulse_score"] = view_scores["motion_impulse_score_raw"].clip(0.0, 1.0).astype(float)
        rows.append(view_scores)

    return pd.concat(rows, ignore_index=True).sort_values(["sample_id", "view", "time"]).reset_index(drop=True) if rows else pd.DataFrame()


def draw_vibration_overlay(ax, windows: pd.DataFrame, color: str) -> None:
    if windows is None or windows.empty:
        return
    for row in windows.itertuples(index=False):
        alpha = float(getattr(row, "vibration_score_alpha", 0.0))
        if alpha > 0.0:
            ax.axvspan(float(row.window_start_sec), float(row.window_end_sec), color=color, alpha=alpha, zorder=0)


def select_metric_windows(vibration_df: pd.DataFrame, sample_id: str, view: str, gy: int, gx: int, metric_key: str) -> pd.DataFrame:
    if vibration_df is None or vibration_df.empty:
        return pd.DataFrame()
    return vibration_df[
        vibration_df["sample_id"].astype(str).eq(str(sample_id))
        & vibration_df["view"].astype(str).eq(str(view))
        & vibration_df["grid_row"].astype(int).eq(int(gy))
        & vibration_df["grid_col"].astype(int).eq(int(gx))
        & vibration_df["metric_key"].astype(str).eq(str(metric_key))
    ].copy()


def save_figure(fig, save_path: str | Path) -> None:
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"saved: {save_path}")
    plt.close(fig)


def plot_metric_3x3(metric_df: pd.DataFrame, vibration_df: pd.DataFrame, sample_id: str, view: str, metric_key: str, save_path: str | Path) -> None:
    save_path = Path(save_path)
    if save_path.exists() and not OVERWRITE_GRAPH_OUTPUTS:
        print(f"skipped existing: {save_path}")
        return
    part = metric_df[metric_df["sample_id"].eq(str(sample_id)) & metric_df["view"].eq(str(view))]
    if part.empty:
        return
    grid_rows, grid_cols = infer_grid_shape(part)
    spec = PLOT_METRIC_SPECS[metric_key]
    accel_spec = FLOW_ACCEL_OVERLAY_SPECS.get(metric_key)
    accel_column = str(accel_spec["column"]) if accel_spec is not None and str(accel_spec["column"]) in part.columns else None
    y_limit_values = [part[metric_key]]
    if accel_column is not None:
        y_limit_values.append(part[accel_column])
    y_limits = tuple(spec.get("fixed_ylim", common_ylim(y_limit_values, nonnegative=bool(spec["nonnegative"]))))
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            cell = part[part["grid_row"].eq(gy) & part["grid_col"].eq(gx)].sort_values("time")
            windows = select_metric_windows(vibration_df, sample_id, view, gy, gx, metric_key)
            draw_vibration_overlay(ax, windows, color=str(spec["overlay_color"]))
            if len(cell):
                ax.plot(cell["time"], cell[metric_key], linewidth=1.0, marker="." if "confidence" in metric_key or metric_key.endswith("consistency") else None, markersize=2.0, label=metric_key, zorder=2)
                if accel_column is not None:
                    ax.plot(
                        cell["time"],
                        cell[accel_column],
                        linewidth=0.9,
                        linestyle="--",
                        alpha=0.85,
                        color=str(accel_spec.get("color", "tab:red")),
                        label=str(accel_spec.get("label", accel_column)),
                        zorder=3,
                    )
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            ax.set_ylim(*y_limits)
            title = f"{gx + 1}x{gy + 1}_{view}"
            if len(windows):
                title += f" max_vib={windows['vibration_score'].max():.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel(spec["label"])
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)
    fig.suptitle(f"{metric_key}: {sample_id} / {view}", y=1.01)
    plt.tight_layout()
    save_figure(fig, save_path)


def plot_broad_vib_score(broad_df: pd.DataFrame, sample_id: str, view: str, broad_key: str, save_path: str | Path) -> None:
    save_path = Path(save_path)
    if save_path.exists() and not OVERWRITE_GRAPH_OUTPUTS:
        print(f"skipped existing: {save_path}")
        return
    part = broad_df[
        broad_df["sample_id"].astype(str).eq(str(sample_id))
        & broad_df["view"].astype(str).eq(str(view))
        & broad_df["broad_output_key"].astype(str).eq(str(broad_key))
    ].sort_values("window_start_sec")
    if part.empty:
        return
    fig, ax = plt.subplots(1, 1, figsize=(12, 3.2))
    ax.plot(part["window_start_sec"], part["broad_vib_score"], linewidth=1.4, marker="o", markersize=2.5, label=broad_key)
    y_values = pd.to_numeric(part["broad_vib_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    y_max = float(np.max(y_values)) if y_values.size else 0.0
    ax.set_ylim(0.0, max(y_max * 1.08, 1e-6))
    ax.set_title(f"{broad_key}: {sample_id} / {view}")
    ax.set_xlabel("window start [sec]")
    ax.set_ylabel("broad vibration score")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    save_figure(fig, save_path)


def plot_accel_impact_score(accel_df: pd.DataFrame, sample_id: str, view: str, output_key: str, save_path: str | Path) -> None:
    save_path = Path(save_path)
    if save_path.exists() and not OVERWRITE_GRAPH_OUTPUTS:
        print(f"skipped existing: {save_path}")
        return
    if accel_df is None or accel_df.empty:
        return
    spec = ACCEL_IMPACT_OUTPUT_SPECS[output_key]
    score_column = str(spec["score_column"])
    part = accel_df[
        accel_df["sample_id"].astype(str).eq(str(sample_id))
        & accel_df["view"].astype(str).eq(str(view))
    ].sort_values("window_start_sec" if "window_start_sec" in accel_df.columns else "time")
    if part.empty or score_column not in part.columns:
        return
    x_column = "window_start_sec" if "window_start_sec" in part.columns else "time"
    fig, ax = plt.subplots(1, 1, figsize=(12, 3.2))
    ax.plot(part[x_column], part[score_column], linewidth=1.4, marker="o", markersize=2.5, label=output_key)
    y_values = pd.to_numeric(part[score_column], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    y_max = float(np.max(y_values)) if y_values.size else 0.0
    ax.set_ylim(0.0, max(y_max * 1.08, 1e-6))
    ax.set_title(f"{output_key}: {sample_id} / {view}")
    ax.set_xlabel("window start [sec]" if x_column == "window_start_sec" else "time [sec]")
    ax.set_ylabel("window accel impact score")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    save_figure(fig, save_path)


def plot_motion_impulse_score(motion_df: pd.DataFrame, sample_id: str, view: str, save_path: str | Path) -> None:
    save_path = Path(save_path)
    if save_path.exists() and not OVERWRITE_GRAPH_OUTPUTS:
        print(f"skipped existing: {save_path}")
        return
    if motion_df is None or motion_df.empty:
        return
    part = motion_df[
        motion_df["sample_id"].astype(str).eq(str(sample_id))
        & motion_df["view"].astype(str).eq(str(view))
    ].sort_values("time")
    if part.empty:
        return
    fig, ax = plt.subplots(1, 1, figsize=(12, 3.6))
    ax.plot(part["time"], part["motion_impulse_score"], linewidth=1.8, marker="o", markersize=2.5, label="motion_impulse_score")
    component_specs = [
        ("motion_impulse_intensity_score", "intensity", "tab:blue"),
        ("motion_impulse_spread_score", "spread", "tab:green"),
        ("motion_impulse_direction_coherence", "direction coherence", "tab:orange"),
    ]
    for column, label, color in component_specs:
        if column in part.columns:
            ax.plot(part["time"], part[column], linewidth=0.9, alpha=0.55, label=label, color=color)
    y_values = pd.to_numeric(part["motion_impulse_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    y_max = float(np.max(y_values)) if y_values.size else 0.0
    ax.set_ylim(0.0, max(1.0, y_max * 1.08, 1e-6))
    ax.set_title(f"motion_impulse_score: {sample_id} / {view}")
    ax.set_xlabel("time [sec]")
    ax.set_ylabel("score / components")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    save_figure(fig, save_path)


def create_feature_graph_outputs(
    features_df: pd.DataFrame,
    output_dir: str | Path,
    sample_output_dirs: dict[str, Path] | None = None,
    output_settings: dict[str, bool] | None = None,
) -> dict[str, Any]:
    output_settings = output_settings or OUTPUT_SETTINGS
    if not any_graph_output_enabled(output_settings):
        print("Feature graph output skipped because all graph outputs are disabled.")
        return {"grid_flow": pd.DataFrame(), "metric_values": pd.DataFrame(), "vibration_scores": pd.DataFrame(), "broad_vib_scores": pd.DataFrame(), "motion_impulse_scores": pd.DataFrame(), "accel_impact_scores": pd.DataFrame()}

    sample_output_dirs = sample_output_dirs or {}
    grid_df = normalize_grid_flow_features(features_df)
    metric_df = add_plot_metrics(grid_df)
    metric_keys = enabled_metric_keys(output_settings)
    vibration_df = build_vibration_scores(metric_df, metric_keys)
    broad_df = build_broad_vib_scores(vibration_df)
    motion_impulse_df = build_motion_impulse_scores(metric_df)
    accel_impact_df = build_accel_impact_scores(metric_df)

    for sample_id, sample_metric_df in metric_df.groupby("sample_id", sort=False):
        sample_dir = Path(sample_output_dirs.get(str(sample_id), Path(output_dir) / safe_path_part(str(sample_id))))
        sample_dir.mkdir(parents=True, exist_ok=True)
        for view in [v for v in ["front", "rear"] if v in set(sample_metric_df["view"].astype(str))]:
            for output_key in OUTPUT_FILE_ORDER:
                if output_key == "raw_flow_video" or not output_enabled(output_key, output_settings):
                    continue
                if output_key in PLOT_METRIC_SPECS:
                    plot_metric_3x3(
                        metric_df,
                        vibration_df,
                        sample_id,
                        view,
                        output_key,
                        numbered_output_path(sample_dir, sample_id, view, output_key, "png"),
                    )
                elif output_key in BROAD_OUTPUT_TO_METRIC:
                    plot_broad_vib_score(
                        broad_df,
                        sample_id,
                        view,
                        output_key,
                        numbered_output_path(sample_dir, sample_id, view, output_key, "png"),
                    )
                elif output_key in ACCEL_IMPACT_OUTPUT_SPECS:
                    plot_accel_impact_score(
                        accel_impact_df,
                        sample_id,
                        view,
                        output_key,
                        numbered_output_path(sample_dir, sample_id, view, output_key, "png"),
                    )
                elif output_key in MOTION_IMPULSE_OUTPUT_KEYS:
                    plot_motion_impulse_score(
                        motion_impulse_df,
                        sample_id,
                        view,
                        numbered_output_path(sample_dir, sample_id, view, output_key, "png"),
                    )
    print(f"saved feature graph outputs under: {output_dir}")
    return {"grid_flow": grid_df, "metric_values": metric_df, "vibration_scores": vibration_df, "broad_vib_scores": broad_df, "motion_impulse_scores": motion_impulse_df, "accel_impact_scores": accel_impact_df}


In [21]:
# ------------------------------------------------------------
# セル概要: 複数対象の一括処理
# ------------------------------------------------------------
# - OUTPUT_SETTINGS の18項目だけを対象に、front/rear の動画・グラフを作成します。
# - CSVや旧グラフなど、指定外のファイル出力は行いません。
# ------------------------------------------------------------

batch_rows = []
feature_dfs = []
sample_output_dirs: dict[str, Path] = {}
graph_outputs_enabled = any_graph_output_enabled(OUTPUT_SETTINGS)
raw_flow_video_enabled = output_enabled("raw_flow_video", OUTPUT_SETTINGS)
need_grid_flow_features = graph_outputs_enabled
runtime_feature_outputs = {"grid_flow": bool(need_grid_flow_features or ENABLE_SAMPLE_FEATURE_CACHE)}


def add_sample_view_columns(features_part: pd.DataFrame, sample_id: str, view: str) -> pd.DataFrame:
    features_part = features_part.copy()
    if "view" in features_part.columns:
        features_part["view"] = view
    else:
        features_part.insert(0, "view", view)
    if "sample_id" in features_part.columns:
        features_part["sample_id"] = sample_id
    else:
        features_part.insert(0, "sample_id", sample_id)
    return features_part


def infer_category_environment(input_video_path: Path) -> tuple[str, str]:
    try:
        category, environment = input_video_path.relative_to(MOVIE_PREPROCESS_DIR).parts[:2]
    except ValueError:
        category, environment = "unknown", "unknown"
    return str(category), str(environment)


def enabled_view_output_paths(sample_dir: Path, sample_id: str, view: str, output_settings: dict[str, bool]) -> list[Path]:
    paths: list[Path] = []
    for output_key in OUTPUT_FILE_ORDER:
        if output_key == "raw_flow_video" or not output_enabled(output_key, output_settings):
            continue
        paths.append(numbered_output_path(sample_dir, sample_id, view, output_key, "png"))
    return paths


def missing_view_output_paths(sample_dir: Path, sample_id: str, view: str, output_settings: dict[str, bool]) -> list[Path]:
    enabled_paths = enabled_view_output_paths(sample_dir, sample_id, view, output_settings)
    if OVERWRITE_GRAPH_OUTPUTS:
        return enabled_paths
    return [path for path in enabled_paths if not path.exists()]


SHARED_FLOW_CACHE_SETTINGS = feature_cache.build_sample_flow_cache_settings(
    cache_version=FLOW_CACHE_VERSION,
    use_front=True,
    use_rear=True,
    flow_sample_sec=float(VISUALIZATION_CONFIG.get("flow_sample_sec", 0.1)),
    frame_resize_width=VISUALIZATION_CONFIG.get("resize_width"),
    flow_analysis_scale=1.0,
    flow_grid=tuple(VISUALIZATION_CONFIG.get("grid", {}).get("shape", (3, 3))),
    flow_reliable_error_threshold_px=float(VISUALIZATION_CONFIG.get("flow_reliable_error_threshold_px", 1.0)),
    flow_method=FLOW_CACHE_METHOD,
)

SHARED_FEATURE_CACHE_SETTINGS = feature_cache.build_feature_cache_settings(
    cache_version=FEATURE_CACHE_VERSION,
    use_front=True,
    use_rear=True,
    flow_sample_sec=float(VISUALIZATION_CONFIG.get("flow_sample_sec", 0.1)),
    window_sec=SHARED_CACHE_WINDOW_SEC,
    audio_sr=SHARED_CACHE_AUDIO_SR,
    n_mels=SHARED_CACHE_N_MELS,
    frame_resize_width=SHARED_CACHE_FRAME_RESIZE_WIDTH,
    flow_analysis_scale=SHARED_CACHE_FLOW_ANALYSIS_SCALE,
    flow_grid=tuple(VISUALIZATION_CONFIG.get("grid", {}).get("shape", (3, 3))),
    flow_reliable_error_threshold_px=float(VISUALIZATION_CONFIG.get("flow_reliable_error_threshold_px", 1.0)),
    flow_score_window_sec=float(VIBRATION_SCORE_CONFIG.get("window_sec", 1.0)),
    flow_score_hop_sec=float(VIBRATION_SCORE_CONFIG.get("hop_sec", 0.5)),
    flow_score_min_visible=1e-6,
    flow_score_high_ratio_fraction=float(VIBRATION_SCORE_CONFIG.get("high_ratio_fraction", 0.5)),
    vibration_score_lower_percentile=0.0,
    vibration_score_upper_percentile=95.0,
    broad_vibration_top_k=None,
    broad_vibration_spread_power=float(VIBRATION_SCORE_CONFIG.get("broad_vib_spread_power", 1.0)),
    broad_vibration_feature_names=SHARED_CACHE_BROAD_VIBRATION_FEATURE_NAMES,
    flow_method=FLOW_CACHE_METHOD,
)
REQUIRED_GRID_FLOW_COLUMNS = {
    "flow_edge_confidence",
    "flow_coherence_confidence",
    "flow_backward_consistency",
    "flow_measurement_confidence",
}


def grid_parts_have_required_columns(grid_parts: dict[str, pd.DataFrame]) -> bool:
    if not grid_parts:
        return False
    for part in grid_parts.values():
        if part is not None and len(part) and not REQUIRED_GRID_FLOW_COLUMNS.issubset(set(part.columns)):
            return False
    return True


def build_shared_feature_cache_sample(sample_id: str, front_video_path: Path, rear_video_path: Path) -> dict[str, Any]:
    category, environment = infer_category_environment(front_video_path)
    audio_path = find_visualization_audio_source(front_video_path)
    return {
        "sample_id": str(sample_id),
        "front_path": str(front_video_path),
        "rear_path": str(rear_video_path),
        "audio_path": str(audio_path) if Path(audio_path).exists() else None,
        "category": category,
        "environment": environment,
        "plot_label": str(sample_id),
    }


def load_shared_feature_cache_grid_parts(sample_id: str, front_video_path: Path, rear_video_path: Path) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    sample = build_shared_feature_cache_sample(sample_id, front_video_path, rear_video_path)
    return feature_cache.load_sample_feature_cache_grid_parts(
        sample,
        settings=SHARED_FEATURE_CACHE_SETTINGS,
        cache_dir=FEATURE_CACHE_DIR,
        views=("front", "rear"),
        enable_cache=ENABLE_SAMPLE_FEATURE_CACHE,
    )


def load_shared_flow_cache_grid_parts(sample_id: str, front_video_path: Path, rear_video_path: Path) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    global feature_cache
    if not hasattr(feature_cache, "load_sample_flow_cache_grid_parts"):
        import importlib as _importlib
        feature_cache = _importlib.reload(feature_cache)
    sample = build_shared_feature_cache_sample(sample_id, front_video_path, rear_video_path)
    return feature_cache.load_sample_flow_cache_grid_parts(
        sample,
        settings=SHARED_FLOW_CACHE_SETTINGS,
        cache_dir=FEATURE_CACHE_DIR,
        views=("front", "rear"),
        enable_cache=ENABLE_SAMPLE_FEATURE_CACHE,
    )


def load_shared_grid_flow_parts(sample_id: str, front_video_path: Path, rear_video_path: Path) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    flow_parts, flow_info = load_shared_flow_cache_grid_parts(sample_id, front_video_path, rear_video_path)
    if flow_parts and grid_parts_have_required_columns(flow_parts):
        return flow_parts, flow_info
    feature_parts, feature_info = load_shared_feature_cache_grid_parts(sample_id, front_video_path, rear_video_path)
    if feature_parts and grid_parts_have_required_columns(feature_parts):
        feature_info = dict(feature_info)
        feature_info["cache_status"] = f"feature_{feature_info.get('cache_status', 'hit')}"
        return feature_parts, feature_info
    if flow_parts or feature_parts:
        flow_info = dict(flow_info)
        flow_info["cache_status"] = "stale_missing_confidence_columns"
    return {}, flow_info


def save_shared_flow_cache_grid_parts(sample_id: str, front_video_path: Path, rear_video_path: Path, grid_parts: dict[str, pd.DataFrame]) -> dict[str, Any]:
    sample = build_shared_feature_cache_sample(sample_id, front_video_path, rear_video_path)
    raw_flow_df = feature_cache.visualization_grid_parts_to_raw_flow(
        grid_parts,
        sample_id=sample_id,
        flow_sample_sec=float(VISUALIZATION_CONFIG.get("flow_sample_sec", 0.1)),
    )
    return feature_cache.save_sample_flow_cache_for_sample(
        sample,
        settings=SHARED_FLOW_CACHE_SETTINGS,
        cache_dir=FEATURE_CACHE_DIR,
        raw_flow_df=raw_flow_df,
        enable_cache=ENABLE_SAMPLE_FEATURE_CACHE and len(raw_flow_df) > 0,
    )


for target in VIDEO_TARGET_PAIRS:
    sample_id = str(target["sample_id"])
    front_video_path = Path(target["front_video_path"])
    rear_video_path = Path(target["rear_video_path"])
    output_dir, front_output_video_path, _ = build_output_paths(front_video_path, sample_id, "front")
    _, rear_output_video_path, _ = build_output_paths(rear_video_path, sample_id, "rear")
    sample_output_dirs[sample_id] = output_dir
    print(f"\n=== {sample_id} / front,rear ===")

    shared_cache_grid_parts, shared_cache_info = load_shared_grid_flow_parts(sample_id, front_video_path, rear_video_path)
    if shared_cache_info.get("cache_status") not in {"miss", "disabled"}:
        print(f"shared flow cache: {shared_cache_info['cache_status']} ({shared_cache_info['cache_path']})")
    sample_grid_parts_for_cache = dict(shared_cache_grid_parts)
    measured_grid_parts: dict[str, pd.DataFrame] = {}

    for view, input_video_path, output_video_path in [
        ("front", front_video_path, front_output_video_path),
        ("rear", rear_video_path, rear_output_video_path),
    ]:
        print(f"--- {sample_id} / {view} ---")
        output_video_path = Path(output_video_path)
        missing_graph_paths = missing_view_output_paths(output_dir, sample_id, view, OUTPUT_SETTINGS) if graph_outputs_enabled else []
        view_needs_graph_features = bool(missing_graph_paths)
        view_needs_raw_video = bool(raw_flow_video_enabled and not output_video_path.exists())
        shared_cache_part = shared_cache_grid_parts.get(view, pd.DataFrame())
        shared_cache_has_grid = len(shared_cache_part) > 0
        features_part = shared_cache_part if view_needs_graph_features and shared_cache_has_grid else None
        status = "loaded_shared_flow_cache" if features_part is not None else "processed"

        if not view_needs_graph_features and not view_needs_raw_video:
            features_part = pd.DataFrame()
            status = "skipped_existing_outputs"
            print(f"skipped existing outputs: {sample_id} / {view}")

        elif view_needs_graph_features and shared_cache_has_grid:
            print(f"using cached grid flow for missing graph outputs: {len(missing_graph_paths)} files")

        if features_part is not None and raw_flow_video_enabled and not output_video_path.exists():
            print("rendering raw flow video from cached grid flow.")
            create_feature_visualization_video(
                input_video_path=input_video_path,
                output_video_path=output_video_path,
                feature_outputs={"grid_flow": False},
                config=VISUALIZATION_CONFIG,
                output_video=True,
                precomputed_grid_flow=shared_cache_part,
            )
            status = f"{status}_rendered_raw_flow_video"

        if features_part is None and (view_needs_graph_features or view_needs_raw_video):
            output_video_enabled = bool(raw_flow_video_enabled and not output_video_path.exists())
            features_part = create_feature_visualization_video(
                input_video_path=input_video_path,
                output_video_path=output_video_path,
                feature_outputs=runtime_feature_outputs,
                config=VISUALIZATION_CONFIG,
                output_video=output_video_enabled,
                precomputed_grid_flow=shared_cache_part if shared_cache_has_grid else None,
            )
            if raw_flow_video_enabled and not output_video_enabled and output_video_path.exists():
                print(f"kept existing raw flow video: {output_video_path}")
            if len(features_part):
                measured_grid_parts[view] = features_part
                sample_grid_parts_for_cache[view] = features_part
        elif features_part is None:
            features_part = pd.DataFrame()
            status = "skipped_disabled_outputs"
            print("scan skipped because all 18 outputs are disabled.")

        features_part = add_sample_view_columns(features_part, sample_id, view)
        feature_dfs.append(features_part)
        batch_rows.append({
            "sample_id": sample_id,
            "view": view,
            "status": status,
            "input_video_path": str(input_video_path),
            "output_dir": str(output_dir),
            "raw_flow_video_path": str(output_video_path) if output_video_path.exists() or raw_flow_video_enabled else None,
            "feature_rows": len(features_part),
        })

    if measured_grid_parts:
        flow_cache_info = save_shared_flow_cache_grid_parts(sample_id, front_video_path, rear_video_path, sample_grid_parts_for_cache)
        print(f"saved shared flow cache: {flow_cache_info['cache_status']} ({flow_cache_info['cache_path']})")

batch_results_df = pd.DataFrame(batch_rows)
features_df = pd.concat(feature_dfs, ignore_index=True) if feature_dfs else pd.DataFrame()
display(batch_results_df)
display(features_df.head())

if graph_outputs_enabled and not features_df.empty:
    feature_graph_results = create_feature_graph_outputs(
        features_df,
        output_dir=OUTPUT_DIR,
        sample_output_dirs=sample_output_dirs,
        output_settings=OUTPUT_SETTINGS,
    )
    if not feature_graph_results["vibration_scores"].empty:
        display(feature_graph_results["vibration_scores"].sort_values("vibration_score", ascending=False).head(30))
    if not feature_graph_results["broad_vib_scores"].empty:
        display(feature_graph_results["broad_vib_scores"].sort_values("broad_vib_score", ascending=False).head(30))
    if not feature_graph_results["motion_impulse_scores"].empty:
        display(feature_graph_results["motion_impulse_scores"].sort_values("motion_impulse_score", ascending=False).head(30))
    if not feature_graph_results["accel_impact_scores"].empty:
        display(feature_graph_results["accel_impact_scores"].sort_values(["accel_impact_x_score", "accel_impact_y_score", "t_accel_impact_x_score", "t_accel_impact_y_score"], ascending=False).head(30))
else:
    feature_graph_results = {}
    print("Feature graph output skipped because graph outputs are disabled or no features were produced.")



=== 032_停車中に車両に衝突される / front,rear ===
shared flow cache: hit (/workspaces/forklift/outputs/anomaly_feature_poc/sample_feature_cache/flow/032_停車中に車両に衝突される_fc5bf63218ea0872b15b.joblib)
--- 032_停車中に車両に衝突される / front ---
using cached grid flow for missing graph outputs: 7 files
--- 032_停車中に車両に衝突される / rear ---
using cached grid flow for missing graph outputs: 7 files

=== 060_後進時、ラックに衝突 / front,rear ===
shared flow cache: hit (/workspaces/forklift/outputs/anomaly_feature_poc/sample_feature_cache/flow/060_後進時_ラックに衝突_34e105ccb5d60cc7c45c.joblib)
--- 060_後進時、ラックに衝突 / front ---
using cached grid flow for missing graph outputs: 7 files
--- 060_後進時、ラックに衝突 / rear ---
using cached grid flow for missing graph outputs: 7 files


,sample_id,view,status,input_video_path,output_dir,raw_flow_video_path,feature_rows
0,032_停車中に車両に衝突される,front,loaded_shared_flow_cache,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,1251
1,032_停車中に車両に衝突される,rear,loaded_shared_flow_cache,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,1251
2,060_後進時、ラックに衝突,front,loaded_shared_flow_cache,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,1260
3,060_後進時、ラックに衝突,rear,loaded_shared_flow_cache,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,1260


,sample_id,view,time,grid_row,grid_col,grid_y,grid_x,flow_x,flow_y,flow_magnitude_mean,...,flow_forward_backward_error_mean,flow_mode,flow_confidence,flow_observed_dt_sec,flow_source_start_sec,flow_source_end_sec,flow_gap_capture_frames,flow_interpolated,flow_hold,flow_update_frame_index
0,032_停車中に車両に衝突される,front,0.0,0,0,0,0,0.001700,0.000539,0.072995,...,0.007357,observed_update,1.0,0.1,0.0,0.1,1.0,0.0,0.0,1.0
1,032_停車中に車両に衝突される,front,0.0,0,1,0,1,0.002458,-0.000854,0.081821,...,0.009956,observed_update,1.0,0.1,0.0,0.1,1.0,0.0,0.0,1.0
2,032_停車中に車両に衝突される,front,0.0,0,2,0,2,0.005164,-0.001673,0.090108,...,0.014644,observed_update,1.0,0.1,0.0,0.1,1.0,0.0,0.0,1.0
3,032_停車中に車両に衝突される,front,0.0,1,0,1,0,0.001450,0.011981,0.060296,...,0.006457,observed_update,1.0,0.1,0.0,0.1,1.0,0.0,0.0,1.0
4,032_停車中に車両に衝突される,front,0.0,1,1,1,1,0.003433,0.000618,0.062679,...,0.006674,observed_update,1.0,0.1,0.0,0.1,1.0,0.0,0.0,1.0


/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 20572 (\N{CJK UNIFIED IDEOGRAPH-505C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 36554 (\N{CJK UNIFIED IDEOGRAPH-8ECA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 20013 (\N{CJK UNIFIED IDEOGRAPH-4E2D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 12395 (\N{HIRAGANA LETTER NI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 20001 (\N{CJK UNIFIED IDEOGRAPH-4E21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 34909 (\N{CJK UNIFIED IDEOGRAPH-885D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_94202/2228621308.py:698: UserWarning: Glyph 31361 (\N{CJK UNIFIED IDEOGRAPH

saved: /workspaces/forklift/outputs/feature_visualization/032_停車中に車両に衝突される_anomaly_indoor/032_停車中に車両に衝突される_front_07_t_flow_x.png


...

saved: /workspaces/forklift/outputs/feature_visualization/060_後進時_ラックに衝突_anomaly_indoor/060_後進時_ラックに衝突_rear_20_flow_backward_consistency.png
saved feature graph outputs under: /workspaces/forklift/outputs/feature_visualization


,sample_id,view,grid_row,grid_col,metric_key,window_start_bin,window_start_sec,window_end_sec,window_center_sec,score_family,...,vibration_score,vibration_score_alpha,sustained_sum,sustained_p95,sustained_high_ratio,sustained_top_mean,sustained_sum_z,sustained_p95_z,sustained_high_ratio_z,sustained_top_mean_z
3817,060_後進時、ラックに衝突,front,0,0,flow_backward_consistency,63,12.600000,13.100000,12.850000,sustained_change,...,1.0,0.42,4.959097,0.993903,1.0,0.993663,2.635172,1.671121,0.0,2.010975
7284,060_後進時、ラックに衝突,rear,2,2,flow_backward_consistency,62,12.400001,12.900001,12.650001,sustained_change,...,1.0,0.42,4.964253,0.997611,1.0,0.997439,1.117002,1.066026,0.0,1.093741
3816,060_後進時、ラックに衝突,front,0,0,flow_backward_consistency,62,12.400001,12.900001,12.650001,sustained_change,...,1.0,0.42,4.974045,0.997076,1.0,0.996806,2.752957,1.791965,0.0,2.140232
4636,060_後進時、ラックに衝突,front,1,1,flow_backward_consistency,66,13.200000,13.700000,13.450000,sustained_change,...,1.0,0.42,4.926181,0.995580,1.0,0.993637,1.642540,2.089647,0.0,2.539163
4653,060_後進時、ラックに衝突,front,1,2,t_flow_x,15,3.000000,3.500000,3.250000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4651,060_後進時、ラックに衝突,front,1,2,t_flow_x,13,2.600000,3.100000,2.850000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4650,060_後進時、ラックに衝突,front,1,2,t_flow_x,12,2.400000,2.900000,2.650000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2654,032_停車中に車両に衝突される,rear,1,1,t_flow_x,41,8.200000,8.700000,8.450000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2653,032_停車中に車両に衝突される,rear,1,1,t_flow_x,40,8.000000,8.500000,8.250000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2652,032_停車中に車両に衝突される,rear,1,1,t_flow_x,39,7.800000,8.300000,8.050000,oscillation,...,1.0,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,sample_id,view,metric_key,broad_output_key,window_start_bin,window_start_sec,window_end_sec,broad_vib_score,mean_vibration_score,mean_change_sum,...,low_intensity_percentile,max_vibration_intensity,spread_score,spread_score_weighted,spread_power,effective_cells,grid_count,broad_weight_a,broad_weight_b,broad_weight_c
308,032_停車中に車両に衝突される,rear,t_flow_x,t_flow_x_broad_vib_score,40,8.000000,8.500000,16.515535,1.000000,23.599223,...,20.0,36.517106,0.854081,0.854081,1.0,7.686732,9,0.25,0.75,0.0
307,032_停車中に車両に衝突される,rear,t_flow_x,t_flow_x_broad_vib_score,39,7.800000,8.300000,10.231467,0.895486,21.016088,...,20.0,36.050486,0.780239,0.780239,1.0,7.022148,9,0.25,0.75,0.0
109,032_停車中に車両に衝突される,front,t_flow_x,t_flow_x_broad_vib_score,42,8.400001,8.900001,9.532728,0.972802,13.837889,...,20.0,25.924066,0.845991,0.845991,1.0,7.613917,9,0.25,0.75,0.0
108,032_停車中に車両に衝突される,front,t_flow_x,t_flow_x_broad_vib_score,41,8.200000,8.700000,8.950473,0.895115,15.031017,...,20.0,37.739553,0.688845,0.688845,1.0,6.199608,9,0.25,0.75,0.0
107,032_停車中に車両に衝突される,front,t_flow_x,t_flow_x_broad_vib_score,40,8.000000,8.500000,8.943886,0.983705,12.314077,...,20.0,29.874064,0.759769,0.759769,1.0,6.837920,9,0.25,0.75,0.0
309,032_停車中に車両に衝突される,rear,t_flow_x,t_flow_x_broad_vib_score,41,8.200000,8.700000,7.735558,0.857676,14.947703,...,20.0,27.582836,0.754688,0.754688,1.0,6.792196,9,0.25,0.75,0.0
310,032_停車中に車両に衝突される,rear,t_flow_x,t_flow_x_broad_vib_score,42,8.400001,8.900001,7.196579,0.970870,14.436422,...,20.0,28.048141,0.765883,0.765883,1.0,6.892951,9,0.25,0.75,0.0
106,032_停車中に車両に衝突される,front,t_flow_x,t_flow_x_broad_vib_score,39,7.800000,8.300000,5.671741,0.893401,10.176532,...,20.0,22.356421,0.751471,0.751471,1.0,6.763241,9,0.25,0.75,0.0
730,060_後進時、ラックに衝突,rear,t_flow_x,t_flow_x_broad_vib_score,56,11.200000,11.700000,5.443854,0.706299,15.542286,...,20.0,22.702470,0.715657,0.715657,1.0,6.440910,9,0.25,0.75,0.0
668,060_後進時、ラックに衝突,rear,flow_backward_consistency,flow_backward_consistency,62,12.400001,12.900001,4.848572,0.994483,4.938108,...,20.0,4.988872,0.999643,0.999643,1.0,8.996790,9,0.25,0.75,0.0


,sample_id,view,time,motion_impulse_intensity_raw,motion_impulse_spread_score,motion_impulse_direction_coherence,motion_impulse_effective_cells,motion_impulse_grid_count,motion_impulse_active_grid_ratio,motion_impulse_vector_change_sum,...,motion_impulse_intensity_weight,motion_impulse_spread_weight,motion_impulse_direction_weight,motion_impulse_intensity_score,motion_impulse_spread_component,motion_impulse_direction_component,motion_impulse_intensity_gate,motion_impulse_component_score,motion_impulse_score_raw,motion_impulse_score
89,032_停車中に車両に衝突される,front,8.9,315.261643,0.876763,0.891181,7.890870,9,1.000000,20.063839,...,0.15,0.45,0.4,1.000000,0.912048,0.922521,1.000000,0.929430,0.929430,0.929430
87,032_停車中に車両に衝突される,front,8.7,676.130071,0.854623,0.891295,7.691608,9,1.000000,42.660602,...,0.15,0.45,0.4,1.000000,0.895864,0.922603,1.000000,0.922180,0.922180,0.922180
221,032_停車中に車両に衝突される,rear,8.2,1263.323445,0.759614,0.998771,6.836524,9,1.000000,94.234896,...,0.15,0.45,0.4,1.000000,0.824926,0.999140,1.000000,0.920873,0.920873,0.920873
226,032_停車中に車両に衝突される,rear,8.7,337.933926,0.742178,0.945612,6.679605,9,1.000000,25.334672,...,0.15,0.45,0.4,1.000000,0.811626,0.961610,1.000000,0.899876,0.899876,0.899876
223,032_停車中に車両に衝突される,rear,8.4,645.792635,0.699748,0.977047,6.297729,9,1.000000,48.194700,...,0.15,0.45,0.4,1.000000,0.778859,0.983877,1.000000,0.894037,0.894037,0.894037
225,032_停車中に車両に衝突される,rear,8.6,266.604192,0.750636,0.911156,6.755720,9,1.000000,23.435119,...,0.15,0.45,0.4,1.000000,0.818089,0.936947,1.000000,0.892919,0.892919,0.892919
83,032_停車中に車両に衝突される,front,8.3,940.343169,0.810731,0.748955,7.296575,9,1.000000,48.082837,...,0.15,0.45,0.4,1.000000,0.863403,0.816806,1.000000,0.865254,0.865254,0.865254
519,060_後進時、ラックに衝突,rear,10.1,11.212068,0.712681,0.843647,6.414128,9,1.000000,57.621186,...,0.15,0.45,0.4,1.000000,0.788908,0.887795,1.000000,0.860127,0.860127,0.860127
82,032_停車中に車両に衝突される,front,8.2,1041.898778,0.735759,0.782753,6.621830,9,1.000000,48.905210,...,0.15,0.45,0.4,1.000000,0.806705,0.842437,1.000000,0.849992,0.849992,0.849992
85,032_停車中に車両に衝突される,front,8.5,594.572365,0.635077,0.762784,5.715694,9,1.000000,39.037506,...,0.15,0.45,0.4,1.000000,0.727744,0.827334,1.000000,0.808418,0.808418,0.808418


,sample_id,view,window_start_bin,window_start_sec,window_end_sec,window_center_sec,time,accel_impact_grid_count,accel_impact_sample_count,accel_impact_x_score,...,t_accel_impact_x_pos_sum,t_accel_impact_x_neg_sum,t_accel_impact_x_clip_abs,t_accel_impact_y_score,signed_t_accel_impact_y_score,t_accel_impact_y_mass,t_accel_impact_y_dominance,t_accel_impact_y_pos_sum,t_accel_impact_y_neg_sum,t_accel_impact_y_clip_abs
41,032_停車中に車両に衝突される,front,41,8.200000,8.700000,8.450000,8.450000,9,5,0.694421,...,4.389661,3.336642,1.0,0.085963,-0.014574,0.354401,0.201853,2.482904,2.700098,1.0
108,032_停車中に車両に衝突される,rear,41,8.200000,8.700000,8.450000,8.450000,9,5,0.664539,...,3.890628,3.859639,1.0,0.384367,-0.239743,0.528028,0.670452,1.692010,3.933299,1.0
42,032_停車中に車両に衝突される,front,42,8.400001,8.900001,8.650001,8.650001,9,5,0.652109,...,4.283292,2.903525,1.0,0.067678,-0.044725,0.292878,0.245065,1.978138,2.447471,1.0
109,032_停車中に車両に衝突される,rear,42,8.400001,8.900001,8.650001,8.650001,9,5,0.637029,...,2.875631,3.968572,1.0,0.192066,0.065236,0.346318,0.532810,2.316936,1.771817,1.0
43,032_停車中に車両に衝突される,front,43,8.600000,9.100000,8.850000,8.850000,9,5,0.619869,...,2.095509,3.807490,1.0,0.053863,-0.033665,0.237367,0.278284,1.665058,1.969446,1.0
255,060_後進時、ラックに衝突,rear,53,10.600000,11.100000,10.850000,10.850000,9,5,0.611673,...,5.728649,0.554460,1.0,0.054978,-0.002839,0.220210,0.252189,1.631931,1.649169,1.0
262,060_後進時、ラックに衝突,rear,60,12.000000,12.500000,12.250000,12.250000,9,5,0.573293,...,0.402950,5.773175,1.0,0.059183,-0.022671,0.190761,0.326160,1.203290,1.475737,1.0
251,060_後進時、ラックに衝突,rear,49,9.800000,10.300000,10.050000,10.050000,9,5,0.571148,...,2.689393,2.454772,1.0,0.181976,-0.089705,0.335335,0.448569,1.578907,2.416282,1.0
261,060_後進時、ラックに衝突,rear,59,11.800000,12.300000,12.050000,12.050000,9,5,0.552183,...,0.602950,5.874789,1.0,0.062347,0.015861,0.216729,0.282074,1.636607,1.436385,1.0
247,060_後進時、ラックに衝突,rear,45,9.000000,9.500000,9.250000,9.250000,9,5,0.547241,...,2.236622,3.662394,1.0,0.058792,0.046277,0.302492,0.198756,2.615068,2.084499,1.0
